# 02_01 — Limpieza de SER tiques

## Objetivo del notebook

Este notebook inicia la limpieza individual de `ser_tiques`, la fuente temporal principal del bloque SER.

El objetivo de esta fase no es calcular la dificultad de aparcar ni generar paneles espacio-temporales. El objetivo es dejar preparada una tabla limpia de eventos de tique, con fechas válidas, duración coherente, claves espaciales básicas y matrícula de parquímetro disponible para validaciones posteriores.

Antes de limpiar, se verifica la estructura real de los ZIP descargados y se comprueba si los archivos internos son homogéneos entre trimestres.

## Referencia documental

La estructura esperada de `ser_tiques` se toma de la documentación oficial del conjunto de datos, guardada en:

`docs/source_docs/ser/ser_tiques/Tiques_de_aparcamiento.pdf`

Según esa documentación, el dataset contiene tiques SER desagregados por tique, con matrícula de parquímetro, fechas de operación, inicio y fin de validez, distrito, barrio, tipo de zona, distintivo ambiental, minutos reservados e importe.

## 1. Configuración inicial

Se detecta automáticamente la raíz del repositorio para que el notebook funcione aunque se ejecute desde la carpeta `notebooks/`.

In [64]:
from pathlib import Path
import glob
import io
import zipfile
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 180)
pd.set_option("display.width", 260)

def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    start = start.resolve()

    for candidate in [start] + list(start.parents):
        if (candidate / "data_catalog.csv").exists():
            return candidate

    raise FileNotFoundError("No se ha encontrado data_catalog.csv subiendo desde el directorio actual.")

ROOT = find_project_root()
DATA_CATALOG = ROOT / "data_catalog.csv"
REPORTS_TABLES = ROOT / "reports" / "tables"

REPORTS_TABLES.mkdir(parents=True, exist_ok=True)

print("Directorio actual del kernel:", Path.cwd())
print("ROOT detectado:", ROOT)
print("Catálogo existe:", DATA_CATALOG.exists())

Directorio actual del kernel: /Users/hugo/TFM_parking_madrid/notebooks
ROOT detectado: /Users/hugo/TFM_parking_madrid
Catálogo existe: True


## 2. Carga del catálogo y localización de ZIP SER

Primero se comprueba que `ser_tiques` está registrado en `data_catalog.csv` y que el patrón `archivo_raw` apunta a ZIP reales.

In [65]:
catalog = pd.read_csv(DATA_CATALOG)

ser_tiques_catalog = catalog[catalog["dataset_id"] == "ser_tiques"].copy()

if ser_tiques_catalog.empty:
    raise ValueError("No se ha encontrado dataset_id='ser_tiques' en data_catalog.csv")

raw_pattern = ser_tiques_catalog["archivo_raw"].iloc[0]
zip_files = sorted(Path(p) for p in glob.glob(str(ROOT / raw_pattern)))

print("Patrón raw:", raw_pattern)
print("ZIP localizados:", len(zip_files))

if not zip_files:
    raise FileNotFoundError(f"No se han encontrado ZIP con el patrón: {raw_pattern}")

pd.DataFrame({
    "zip_file": [str(p.relative_to(ROOT)) for p in zip_files],
    "size_mb": [round(p.stat().st_size / (1024**2), 3) for p in zip_files],
})

Patrón raw: data/raw/ser/ser_tiques/ser_tiques__*.zip
ZIP localizados: 13


,zip_file,size_mb
0,data/raw/ser/ser_tiques/ser_tiques__2023_q1.zip,215.934
1,data/raw/ser/ser_tiques/ser_tiques__2023_q2.zip,159.087
2,data/raw/ser/ser_tiques/ser_tiques__2023_q3.zip,199.946
3,data/raw/ser/ser_tiques/ser_tiques__2023_q4.zip,199.352
4,data/raw/ser/ser_tiques/ser_tiques__2024_q1.zip,223.511
5,data/raw/ser/ser_tiques/ser_tiques__2024_q2.zip,230.684
6,data/raw/ser/ser_tiques/ser_tiques__2024_q3.zip,141.330
7,data/raw/ser/ser_tiques/ser_tiques__2024_q4.zip,232.798
8,data/raw/ser/ser_tiques/ser_tiques__2025_q1.zip,229.464
9,data/raw/ser/ser_tiques/ser_tiques__2025_q2.zip,225.855


### Lectura inicial

Se confirma que la fuente está completa físicamente antes de intentar leerla. No falta ningún ZIP.

## 3. Inventario interno de los ZIP

Se inspecciona el contenido interno de cada ZIP sin extraerlo en `data/raw`. El directorio `raw` debe conservar los archivos originales sin modificar.

Esta inspección identifica:

- nombre de los archivos internos;
- extensiones internas;
- tamaño interno;
- ZIP con CSV directamente legible;
- casos especiales que requieren tratamiento aparte, como archivos `.rar` dentro de ZIP.

In [66]:
zip_rows = []

for zip_path in zip_files:
    with zipfile.ZipFile(zip_path) as z:
        members = z.infolist()

        for member in members:
            if member.is_dir():
                continue

            member_path = Path(member.filename)
            zip_rows.append({
                "zip_file": str(zip_path.relative_to(ROOT)),
                "zip_size_mb": round(zip_path.stat().st_size / (1024**2), 3),
                "member_name": member.filename,
                "member_extension": member_path.suffix.lower().replace(".", ""),
                "member_size_mb": round(member.file_size / (1024**2), 3),
                "is_csv": member_path.suffix.lower() == ".csv",
                "is_rar": member_path.suffix.lower() == ".rar",
            })

zip_inventory = pd.DataFrame(zip_rows)

zip_inventory_path = REPORTS_TABLES / "ser_tiques_zip_inventory.csv"
zip_inventory.to_csv(zip_inventory_path, index=False)

print("Archivos internos inventariados:", len(zip_inventory))
zip_inventory

Archivos internos inventariados: 13


,zip_file,zip_size_mb,member_name,member_extension,member_size_mb,is_csv,is_rar
0,data/raw/ser/ser_tiques/ser_tiques__2023_q1.zip,215.934,1er 2023 TEMPO+NP.csv,csv,1313.069,True,False
1,data/raw/ser/ser_tiques/ser_tiques__2023_q2.zip,159.087,2otrimestre.csv,csv,1285.795,True,False
2,data/raw/ser/ser_tiques/ser_tiques__2023_q3.zip,199.946,tercertrimestre2023.csv,csv,1048.156,True,False
3,data/raw/ser/ser_tiques/ser_tiques__2023_q4.zip,199.352,4otrimestre.rar,rar,199.291,False,True
4,data/raw/ser/ser_tiques/ser_tiques__2024_q1.zip,223.511,primertrimestre2024.csv,csv,1334.199,True,False
5,data/raw/ser/ser_tiques/ser_tiques__2024_q2.zip,230.684,segundotrimestre2024.csv,csv,1377.068,True,False
6,data/raw/ser/ser_tiques/ser_tiques__2024_q3.zip,141.330,tercertrimestre2024.csv,csv,871.396,True,False
7,data/raw/ser/ser_tiques/ser_tiques__2024_q4.zip,232.798,cuartotrimestre.csv,csv,1388.310,True,False
8,data/raw/ser/ser_tiques/ser_tiques__2025_q1.zip,229.464,Primertrimestre2025.csv,csv,1381.564,True,False
9,data/raw/ser/ser_tiques/ser_tiques__2025_q2.zip,225.855,Segundotrimestre2025.csv,csv,1340.443,True,False


### Incidencia de empaquetado en 2023 Q4

El ZIP `ser_tiques__2023_q4.zip` no contiene directamente un CSV, sino un archivo `.rar`. Para poder incluir ese trimestre en las comprobaciones de estructura, se extrae manualmente el CSV con Keka y se guarda en:

`data/raw/ser/ser_tiques/_extracted_manual/ser_tiques__2023_q4.csv`

Este archivo no es una transformación ni una limpieza: es una extracción controlada de un contenedor comprimido anómalo. Se conserva dentro de `data/raw`, que no se versiona en Git.

In [67]:
manual_extracted_q4 = ROOT / "data/raw/ser/ser_tiques/_extracted_manual/ser_tiques__2023_q4.csv"

print("CSV extraído manualmente existe:", manual_extracted_q4.exists())

if manual_extracted_q4.exists():
    sample_q4 = pd.read_csv(
        manual_extracted_q4,
        sep=None,
        engine="python",
        nrows=5,
        encoding_errors="replace"
    )

    print("Shape muestra 2023 Q4 extraído:", sample_q4.shape)
    print("Columnas 2023 Q4 extraído:")
    for col in sample_q4.columns:
        print("-", repr(col))
else:
    print("No se ha encontrado el CSV extraído manualmente. 2023 Q4 quedará pendiente.")

CSV extraído manualmente existe: True
Shape muestra 2023 Q4 extraído: (5, 12)
Columnas 2023 Q4 extraído:
- '\ufeffmatricula_parquimetro'
- 'fecha_operacion'
- 'fecha_inicio'
- 'fecha_fin'
- 'cod_distrito'
- 'distrito'
- 'cod_barrio'
- 'barrio'
- 'tipo_zona'
- 'distintivo'
- 'minutos_tique'
- 'importe_tique'


## 4. Lectura muestral de columnas internas

Se lee una muestra pequeña de cada CSV interno para identificar columnas reales, separador y problemas básicos de lectura.

No se carga el archivo completo. Esta lectura solo sirve para decidir si los trimestres tienen una estructura homogénea.

In [68]:
def read_internal_csv_sample(zip_path, member_name, nrows=5, max_bytes=2_000_000):
    # Lee solo una muestra de bytes del CSV interno.
    # Se prueban codificaciones habituales en ficheros municipales.
    encodings = ["utf-8-sig", "utf-8", "latin1", "cp1252"]

    with zipfile.ZipFile(zip_path) as z:
        with z.open(member_name) as f:
            data = f.read(max_bytes)

    last_error = None

    for enc in encodings:
        try:
            sample = pd.read_csv(
                io.BytesIO(data),
                sep=None,
                engine="python",
                nrows=nrows,
                encoding=enc,
                encoding_errors="replace"
            )
            return sample, enc, None
        except Exception as exc:
            last_error = repr(exc)

    return None, None, last_error


def read_external_csv_sample(path, nrows=5):
    encodings = ["utf-8-sig", "utf-8", "latin1", "cp1252"]
    last_error = None

    for enc in encodings:
        try:
            sample = pd.read_csv(
                path,
                sep=None,
                engine="python",
                nrows=nrows,
                encoding=enc,
                encoding_errors="replace"
            )
            return sample, enc, None
        except Exception as exc:
            last_error = repr(exc)

    return None, None, last_error


schema_rows = []

for _, row in zip_inventory.iterrows():
    zip_path = ROOT / row["zip_file"]

    # Caso especial: 2023 Q4 viene como .rar dentro del ZIP.
    # Si existe CSV extraído manualmente, se incorpora a la comparación.
    if row["is_rar"] and "ser_tiques__2023_q4.zip" in row["zip_file"]:
        manual_path = ROOT / "data/raw/ser/ser_tiques/_extracted_manual/ser_tiques__2023_q4.csv"

        if manual_path.exists():
            sample, encoding_used, error = read_external_csv_sample(manual_path)

            if sample is None:
                schema_rows.append({
                    "periodo_raw": "2023_q4",
                    "source_type": "manual_extracted_from_rar",
                    "zip_file": row["zip_file"],
                    "member_name": row["member_name"],
                    "effective_file": str(manual_path.relative_to(ROOT)),
                    "member_extension": "csv",
                    "read_ok": False,
                    "encoding_used": encoding_used,
                    "n_columns": None,
                    "columns": None,
                    "error": error,
                })
            else:
                schema_rows.append({
                    "periodo_raw": "2023_q4",
                    "source_type": "manual_extracted_from_rar",
                    "zip_file": row["zip_file"],
                    "member_name": row["member_name"],
                    "effective_file": str(manual_path.relative_to(ROOT)),
                    "member_extension": "csv",
                    "read_ok": True,
                    "encoding_used": encoding_used,
                    "n_columns": sample.shape[1],
                    "columns": "; ".join(map(str, sample.columns.tolist())),
                    "error": None,
                })

        else:
            schema_rows.append({
                "periodo_raw": "2023_q4",
                "source_type": "rar_without_extracted_csv",
                "zip_file": row["zip_file"],
                "member_name": row["member_name"],
                "effective_file": None,
                "member_extension": "rar",
                "read_ok": False,
                "encoding_used": None,
                "n_columns": None,
                "columns": None,
                "error": "RAR interno sin CSV extraído manualmente",
            })

        continue

    if not row["is_csv"]:
        schema_rows.append({
            "periodo_raw": None,
            "source_type": "non_csv_internal",
            "zip_file": row["zip_file"],
            "member_name": row["member_name"],
            "effective_file": None,
            "member_extension": row["member_extension"],
            "read_ok": False,
            "encoding_used": None,
            "n_columns": None,
            "columns": None,
            "error": "No es CSV interno; requiere tratamiento específico",
        })
        continue

    sample, encoding_used, error = read_internal_csv_sample(zip_path, row["member_name"])

    periodo_raw = Path(row["zip_file"]).stem.replace("ser_tiques__", "")

    if sample is None:
        schema_rows.append({
            "periodo_raw": periodo_raw,
            "source_type": "zip_internal_csv",
            "zip_file": row["zip_file"],
            "member_name": row["member_name"],
            "effective_file": row["zip_file"],
            "member_extension": row["member_extension"],
            "read_ok": False,
            "encoding_used": encoding_used,
            "n_columns": None,
            "columns": None,
            "error": error,
        })
    else:
        schema_rows.append({
            "periodo_raw": periodo_raw,
            "source_type": "zip_internal_csv",
            "zip_file": row["zip_file"],
            "member_name": row["member_name"],
            "effective_file": row["zip_file"],
            "member_extension": row["member_extension"],
            "read_ok": True,
            "encoding_used": encoding_used,
            "n_columns": sample.shape[1],
            "columns": "; ".join(map(str, sample.columns.tolist())),
            "error": None,
        })

ser_tiques_schema_summary = pd.DataFrame(schema_rows)

schema_summary_path = REPORTS_TABLES / "ser_tiques_schema_summary.csv"
ser_tiques_schema_summary.to_csv(schema_summary_path, index=False)

ser_tiques_schema_summary

,periodo_raw,source_type,zip_file,member_name,effective_file,member_extension,read_ok,encoding_used,n_columns,columns,error
0,2023_q1,zip_internal_csv,data/raw/ser/ser_tiques/ser_tiques__2023_q1.zip,1er 2023 TEMPO+NP.csv,data/raw/ser/ser_tiques/ser_tiques__2023_q1.zip,csv,True,utf-8-sig,12,matricula_parquimetro; fecha_operacion; fecha_inicio; fecha_fin; cod_distrito; distrito; cod_barrio; barrio; tipo_zona; distintivo; minutos_tique; importe_tique,None
1,2023_q2,zip_internal_csv,data/raw/ser/ser_tiques/ser_tiques__2023_q2.zip,2otrimestre.csv,data/raw/ser/ser_tiques/ser_tiques__2023_q2.zip,csv,True,utf-8-sig,12,matricula_parquimetro; fecha_operacion; fecha_inicio; fecha_fin; cod_distrito; distrito; cod_barrio; barrio; tipo_zona; distintivo; minutos_tique; importe_tique,None
2,2023_q3,zip_internal_csv,data/raw/ser/ser_tiques/ser_tiques__2023_q3.zip,tercertrimestre2023.csv,data/raw/ser/ser_tiques/ser_tiques__2023_q3.zip,csv,True,utf-8-sig,12,matricula_parquimetro; fecha_operacion; fecha_inicio; fecha_fin; cod_distrito; distrito; cod_barrio; barrio; tipo_zona; distintivo; minutos_tique; importe_tique,None
3,2023_q4,manual_extracted_from_rar,data/raw/ser/ser_tiques/ser_tiques__2023_q4.zip,4otrimestre.rar,data/raw/ser/ser_tiques/_extracted_manual/ser_tiques__2023_q4.csv,csv,True,utf-8-sig,12,matricula_parquimetro; fecha_operacion; fecha_inicio; fecha_fin; cod_distrito; distrito; cod_barrio; barrio; tipo_zona; distintivo; minutos_tique; importe_tique,None
4,2024_q1,zip_internal_csv,data/raw/ser/ser_tiques/ser_tiques__2024_q1.zip,primertrimestre2024.csv,data/raw/ser/ser_tiques/ser_tiques__2024_q1.zip,csv,True,utf-8-sig,12,matricula_parquimetro; fecha_operacion; fecha_inicio; fecha_fin; cod_distrito; distrito; cod_barrio; barrio; tipo_zona; distintivo; minutos_tique; importe_tique,None
5,2024_q2,zip_internal_csv,data/raw/ser/ser_tiques/ser_tiques__2024_q2.zip,segundotrimestre2024.csv,data/raw/ser/ser_tiques/ser_tiques__2024_q2.zip,csv,True,utf-8-sig,12,matricula_parquimetro; fecha_operacion; fecha_inicio; fecha_fin; cod_distrito; distrito; cod_barrio; barrio; tipo_zona; distintivo; minutos_tique; importe_tique,None
6,2024_q3,zip_internal_csv,data/raw/ser/ser_tiques/ser_tiques__2024_q3.zip,tercertrimestre2024.csv,data/raw/ser/ser_tiques/ser_tiques__2024_q3.zip,csv,True,utf-8-sig,12,matricula_parquimetro; fecha_operacion; fecha_inicio; fecha_fin; cod_distrito; distrito; cod_barrio; barrio; tipo_zona; distintivo; minutos_tique; importe_tique,None
7,2024_q4,zip_internal_csv,data/raw/ser/ser_tiques/ser_tiques__2024_q4.zip,cuartotrimestre.csv,data/raw/ser/ser_tiques/ser_tiques__2024_q4.zip,csv,True,utf-8-sig,12,matricula_parquimetro; fecha_operacion; fecha_inicio; fecha_fin; cod_distrito; distrito; cod_barrio; barrio; tipo_zona; distintivo; minutos_tique; importe_tique,None
8,2025_q1,zip_internal_csv,data/raw/ser/ser_tiques/ser_tiques__2025_q1.zip,Primertrimestre2025.csv,data/raw/ser/ser_tiques/ser_tiques__2025_q1.zip,csv,True,utf-8-sig,12,matricula_parquimetro; fecha_operacion; fecha_inicio; fecha_fin; cod_distrito; distrito; cod_barrio; barrio; tipo_zona; distintivo; minutos_tique; importe_tique,None
9,2025_q2,zip_internal_csv,data/raw/ser/ser_tiques/ser_tiques__2025_q2.zip,Segundotrimestre2025.csv,data/raw/ser/ser_tiques/ser_tiques__2025_q2.zip,csv,True,utf-8-sig,12,matricula_parquimetro; fecha_operacion; fecha_inicio; fecha_fin; cod_distrito; distrito; cod_barrio; barrio; tipo_zona; distintivo; minutos_tique; importe_tique,None


### Lectura de la inspección de columnas

La inspección incorpora 13 periodos: 12 CSV leídos directamente desde ZIP y el cuarto trimestre de 2023 incorporado desde el CSV extraído manualmente del `.rar`.

La incidencia de 2023 Q4 se interpreta como un problema de empaquetado, no como un problema de estructura analítica. La muestra extraída tiene las mismas 12 columnas relevantes que el resto de trimestres.

Si la comparación posterior confirma que los nombres son equivalentes, la limpieza podrá plantearse como una limpieza conjunta por lotes trimestrales, sin cargar toda la fuente completa en memoria.

## 5. Comparación contra columnas esperadas

Se comparan las columnas reales con las esperadas a partir de la documentación oficial y del checklist operativo.

Esta comparación no decide todavía qué columnas finales se conservarán, pero identifica qué campos están disponibles y cuáles requieren normalización de nombres.

In [69]:
expected_pdf_columns = [
    "matricula_parquimetro",
    "fecha_operacion",
    "fecha_inicio",
    "fecha_fin",
    "codigo_distrito",
    "distrito",
    "codigo_barrio",
    "barrio",
    "tipo_zona",
    "distintivo",
    "minutos_tique",
    "importe_tique",
]

def normalize_column_name(col):
    col = str(col)
    col = col.replace("\ufeff", "")
    col = col.strip().lower()
    col = col.replace("á", "a").replace("é", "e").replace("í", "i").replace("ó", "o").replace("ú", "u")
    col = col.replace("ñ", "n")
    col = col.replace(" ", "_")
    col = col.replace("-", "_")
    col = col.replace("/", "_")
    col = col.replace(".", "")
    while "__" in col:
        col = col.replace("__", "_")
    return col

def normalize_column_set(columns_text):
    if pd.isna(columns_text) or columns_text is None:
        return []
    return [normalize_column_name(c) for c in str(columns_text).split(";")]

expected_norm = set(normalize_column_name(c) for c in expected_pdf_columns)

comparison_rows = []

for _, row in ser_tiques_schema_summary.iterrows():
    actual_cols = normalize_column_set(row["columns"])
    actual_set = set(actual_cols)

    comparison_rows.append({
        "zip_file": row["zip_file"],
        "member_name": row["member_name"],
        "read_ok": row["read_ok"],
        "n_columns": row["n_columns"],
        "missing_expected_columns": "; ".join(sorted(expected_norm - actual_set)),
        "extra_columns": "; ".join(sorted(actual_set - expected_norm)),
        "actual_columns_norm": "; ".join(actual_cols),
    })

ser_tiques_expected_columns_check = pd.DataFrame(comparison_rows)

expected_check_path = REPORTS_TABLES / "ser_tiques_expected_columns_check.csv"
ser_tiques_expected_columns_check.to_csv(expected_check_path, index=False)

ser_tiques_expected_columns_check

,zip_file,member_name,read_ok,n_columns,missing_expected_columns,extra_columns,actual_columns_norm
0,data/raw/ser/ser_tiques/ser_tiques__2023_q1.zip,1er 2023 TEMPO+NP.csv,True,12,codigo_barrio; codigo_distrito,cod_barrio; cod_distrito,matricula_parquimetro; fecha_operacion; fecha_inicio; fecha_fin; cod_distrito; distrito; cod_barrio; barrio; tipo_zona; distintivo; minutos_tique; importe_tique
1,data/raw/ser/ser_tiques/ser_tiques__2023_q2.zip,2otrimestre.csv,True,12,codigo_barrio; codigo_distrito,cod_barrio; cod_distrito,matricula_parquimetro; fecha_operacion; fecha_inicio; fecha_fin; cod_distrito; distrito; cod_barrio; barrio; tipo_zona; distintivo; minutos_tique; importe_tique
2,data/raw/ser/ser_tiques/ser_tiques__2023_q3.zip,tercertrimestre2023.csv,True,12,codigo_barrio; codigo_distrito,cod_barrio; cod_distrito,matricula_parquimetro; fecha_operacion; fecha_inicio; fecha_fin; cod_distrito; distrito; cod_barrio; barrio; tipo_zona; distintivo; minutos_tique; importe_tique
3,data/raw/ser/ser_tiques/ser_tiques__2023_q4.zip,4otrimestre.rar,True,12,codigo_barrio; codigo_distrito,cod_barrio; cod_distrito,matricula_parquimetro; fecha_operacion; fecha_inicio; fecha_fin; cod_distrito; distrito; cod_barrio; barrio; tipo_zona; distintivo; minutos_tique; importe_tique
4,data/raw/ser/ser_tiques/ser_tiques__2024_q1.zip,primertrimestre2024.csv,True,12,codigo_barrio; codigo_distrito,cod_barrio; cod_distrito,matricula_parquimetro; fecha_operacion; fecha_inicio; fecha_fin; cod_distrito; distrito; cod_barrio; barrio; tipo_zona; distintivo; minutos_tique; importe_tique
5,data/raw/ser/ser_tiques/ser_tiques__2024_q2.zip,segundotrimestre2024.csv,True,12,codigo_barrio; codigo_distrito,cod_barrio; cod_distrito,matricula_parquimetro; fecha_operacion; fecha_inicio; fecha_fin; cod_distrito; distrito; cod_barrio; barrio; tipo_zona; distintivo; minutos_tique; importe_tique
6,data/raw/ser/ser_tiques/ser_tiques__2024_q3.zip,tercertrimestre2024.csv,True,12,codigo_barrio; codigo_distrito,cod_barrio; cod_distrito,matricula_parquimetro; fecha_operacion; fecha_inicio; fecha_fin; cod_distrito; distrito; cod_barrio; barrio; tipo_zona; distintivo; minutos_tique; importe_tique
7,data/raw/ser/ser_tiques/ser_tiques__2024_q4.zip,cuartotrimestre.csv,True,12,codigo_barrio; codigo_distrito,cod_barrio; cod_distrito,matricula_parquimetro; fecha_operacion; fecha_inicio; fecha_fin; cod_distrito; distrito; cod_barrio; barrio; tipo_zona; distintivo; minutos_tique; importe_tique
8,data/raw/ser/ser_tiques/ser_tiques__2025_q1.zip,Primertrimestre2025.csv,True,12,codigo_barrio; codigo_distrito,cod_barrio; cod_distrito,matricula_parquimetro; fecha_operacion; fecha_inicio; fecha_fin; cod_distrito; distrito; cod_barrio; barrio; tipo_zona; distintivo; minutos_tique; importe_tique
9,data/raw/ser/ser_tiques/ser_tiques__2025_q2.zip,Segundotrimestre2025.csv,True,12,codigo_barrio; codigo_distrito,cod_barrio; cod_distrito,matricula_parquimetro; fecha_operacion; fecha_inicio; fecha_fin; cod_distrito; distrito; cod_barrio; barrio; tipo_zona; distintivo; minutos_tique; importe_tique


### Lectura de la comparación con la documentación

La comparación muestra que los trimestres legibles contienen las variables necesarias para la limpieza de `ser_tiques`.

La diferencia principal frente a la documentación oficial es de nomenclatura: el PDF usa `codigo_distrito` y `codigo_barrio`, mientras que los archivos reales usan `cod_distrito` y `cod_barrio`. No parece una pérdida de información, sino un caso de renombrado.

## 6. Diagnóstico de estructura raw

La revisión inicial confirma que `ser_tiques` puede limpiarse con una estrategia común por lotes trimestrales.

Los 13 periodos localizados presentan el mismo conjunto de 12 columnas útiles. Doce periodos se leen directamente desde CSV internos en ZIP y el cuarto trimestre de 2023 se incorpora desde el CSV extraído manualmente del `.rar`.

La única incidencia detectada hasta este punto no afecta al esquema analítico, sino al empaquetado del archivo de 2023 Q4. Tras la extracción manual controlada, ese trimestre mantiene la misma estructura que el resto.

La diferencia frente a la documentación oficial es únicamente de nomenclatura: los archivos reales usan `cod_distrito` y `cod_barrio`, mientras que la documentación describe `codigo_distrito` y `codigo_barrio`. Esta diferencia se resolverá mediante renombrado.

Con esta evidencia, la limpieza puede avanzar mediante un diccionario de normalización de columnas y una lectura por lotes trimestrales. La decisión definitiva queda condicionada a que el recuento completo y las validaciones de fechas/duración no detecten anomalías graves.

## 7. Limpieza incremental de SER tiques

En esta sección se construye la salida limpia de `ser_tiques` a partir de los archivos trimestrales. La limpieza se realiza por lotes para controlar memoria y conservar trazabilidad por archivo de origen.

### 7.1. Diccionario de renombrado y columnas esperadas

Una vez comprobado que los 13 periodos tienen el mismo conjunto de columnas útiles, se define el esquema canónico de trabajo.

La documentación oficial usa nombres como `codigo_distrito` y `codigo_barrio`, mientras que los archivos reales usan `cod_distrito` y `cod_barrio`. Para la limpieza se adopta la nomenclatura corta, porque es la que aparece de forma consistente en los datos reales y es la que usan el resto de los datasets del catalogo.

El diccionario también contempla pequeñas variantes posibles, como mayúsculas/minúsculas, espacios, marcas BOM o erratas documentales.

In [70]:
# Columnas canónicas de entrada para ser_tiques.
# No incluyen todavía columnas derivadas como duración calculada o flags de calidad.

SER_TIQUES_REQUIRED_COLUMNS = [
    "matricula_parquimetro",
    "fecha_operacion",
    "fecha_inicio",
    "fecha_fin",
    "cod_distrito",
    "distrito",
    "cod_barrio",
    "barrio",
    "tipo_zona",
    "distintivo",
    "minutos_tique",
    "importe_tique",
]

# Mapa de alias hacia nombres canónicos.
# Se aplica después de normalizar nombres con normalize_column_name().

SER_TIQUES_RENAME_MAP = {
    # Identificador de parquímetro
    "matricula_parquimetro": "matricula_parquimetro",

    # Fechas
    "fecha_operacion": "fecha_operacion",
    "fecha_inicio": "fecha_inicio",
    "fecha_fin": "fecha_fin",
    "feha_fin": "fecha_fin",  # posible errata documental

    # Distrito y barrio
    "codigo_distrito": "cod_distrito",
    "cod_distrito": "cod_distrito",
    "distrito": "distrito",
    "codigo_barrio": "cod_barrio",
    "cod_barrio": "cod_barrio",
    "barrio": "barrio",

    # Características del tique
    "tipo_zona": "tipo_zona",
    "distintivo": "distintivo",
    "minutos_tique": "minutos_tique",
    "importe_tique": "importe_tique",
}


def canonicalize_ser_tiques_column(col: str) -> str:
    """Normaliza una columna raw de SER tiques al nombre canónico del proyecto."""
    col_norm = normalize_column_name(col)
    return SER_TIQUES_RENAME_MAP.get(col_norm, col_norm)


def canonicalize_columns_from_text(columns_text):
    if pd.isna(columns_text) or columns_text is None:
        return []
    return [canonicalize_ser_tiques_column(c) for c in str(columns_text).split(";")]


column_check_rows = []

required_set = set(SER_TIQUES_REQUIRED_COLUMNS)

for _, row in ser_tiques_schema_summary.iterrows():
    canonical_cols = canonicalize_columns_from_text(row["columns"])
    canonical_set = set(canonical_cols)

    column_check_rows.append({
        "periodo_raw": row["periodo_raw"],
        "source_type": row["source_type"],
        "read_ok": row["read_ok"],
        "n_columns_raw": row["n_columns"],
        "n_columns_canonical": len(canonical_cols),
        "missing_required_columns": "; ".join(sorted(required_set - canonical_set)),
        "extra_columns_after_canonicalization": "; ".join(sorted(canonical_set - required_set)),
        "canonical_columns": "; ".join(canonical_cols),
    })

ser_tiques_canonical_columns_check = pd.DataFrame(column_check_rows)

canonical_check_path = REPORTS_TABLES / "ser_tiques_canonical_columns_check.csv"
ser_tiques_canonical_columns_check.to_csv(canonical_check_path, index=False)

display(ser_tiques_canonical_columns_check)

n_missing = (
    ser_tiques_canonical_columns_check["missing_required_columns"]
    .fillna("")
    .ne("")
    .sum()
)

n_extra = (
    ser_tiques_canonical_columns_check["extra_columns_after_canonicalization"]
    .fillna("")
    .ne("")
    .sum()
)

print("Periodos revisados:", len(ser_tiques_canonical_columns_check))
print("Periodos con columnas requeridas faltantes:", n_missing)
print("Periodos con columnas extra tras canonización:", n_extra)

if n_missing == 0 and n_extra == 0:
    print("Resultado: el esquema de columnas queda homogeneizado para todos los periodos.")
else:
    print("Resultado: hay diferencias de columnas que deben revisarse antes de limpiar.")

,periodo_raw,source_type,read_ok,n_columns_raw,n_columns_canonical,missing_required_columns,extra_columns_after_canonicalization,canonical_columns
0,2023_q1,zip_internal_csv,True,12,12,,,matricula_parquimetro; fecha_operacion; fecha_inicio; fecha_fin; cod_distrito; distrito; cod_barrio; barrio; tipo_zona; distintivo; minutos_tique; importe_tique
1,2023_q2,zip_internal_csv,True,12,12,,,matricula_parquimetro; fecha_operacion; fecha_inicio; fecha_fin; cod_distrito; distrito; cod_barrio; barrio; tipo_zona; distintivo; minutos_tique; importe_tique
2,2023_q3,zip_internal_csv,True,12,12,,,matricula_parquimetro; fecha_operacion; fecha_inicio; fecha_fin; cod_distrito; distrito; cod_barrio; barrio; tipo_zona; distintivo; minutos_tique; importe_tique
3,2023_q4,manual_extracted_from_rar,True,12,12,,,matricula_parquimetro; fecha_operacion; fecha_inicio; fecha_fin; cod_distrito; distrito; cod_barrio; barrio; tipo_zona; distintivo; minutos_tique; importe_tique
4,2024_q1,zip_internal_csv,True,12,12,,,matricula_parquimetro; fecha_operacion; fecha_inicio; fecha_fin; cod_distrito; distrito; cod_barrio; barrio; tipo_zona; distintivo; minutos_tique; importe_tique
5,2024_q2,zip_internal_csv,True,12,12,,,matricula_parquimetro; fecha_operacion; fecha_inicio; fecha_fin; cod_distrito; distrito; cod_barrio; barrio; tipo_zona; distintivo; minutos_tique; importe_tique
6,2024_q3,zip_internal_csv,True,12,12,,,matricula_parquimetro; fecha_operacion; fecha_inicio; fecha_fin; cod_distrito; distrito; cod_barrio; barrio; tipo_zona; distintivo; minutos_tique; importe_tique
7,2024_q4,zip_internal_csv,True,12,12,,,matricula_parquimetro; fecha_operacion; fecha_inicio; fecha_fin; cod_distrito; distrito; cod_barrio; barrio; tipo_zona; distintivo; minutos_tique; importe_tique
8,2025_q1,zip_internal_csv,True,12,12,,,matricula_parquimetro; fecha_operacion; fecha_inicio; fecha_fin; cod_distrito; distrito; cod_barrio; barrio; tipo_zona; distintivo; minutos_tique; importe_tique
9,2025_q2,zip_internal_csv,True,12,12,,,matricula_parquimetro; fecha_operacion; fecha_inicio; fecha_fin; cod_distrito; distrito; cod_barrio; barrio; tipo_zona; distintivo; minutos_tique; importe_tique


Periodos revisados: 13
Periodos con columnas requeridas faltantes: 0
Periodos con columnas extra tras canonización: 0
Resultado: el esquema de columnas queda homogeneizado para todos los periodos.


### 7.2. Esquema base, columnas conservadas y columnas descartadas

A partir del esquema canónico de entrada se define una tabla base con las 12 columnas operativas originales normalizadas. En esta fase no se añaden columnas artificiales de trazabilidad (`dataset_id`, `periodo_raw`, `archivo_origen`, `source_type`) al contenido físico de los Parquet, porque la trazabilidad queda cubierta por el catálogo, los manifiestos y la estructura de carpetas por periodo.

La documentación oficial describe que `ser_tiques` contiene tiques expedidos en parquímetros y aplicaciones móviles, desagregados por tique, con matrícula del parquímetro, fechas de operación, inicio y fin de validez, distrito, barrio, tipo de zona, distintivo, minutos reservados e importe.

#### Columnas usadas durante validación

Se mantienen temporalmente las 12 columnas operativas:

- `matricula_parquimetro`: identificador del parquímetro; útil para validación posterior con la fuente de parquímetros.
- `fecha_operacion`: finalización de la obtención del tique; se usa solo como diagnóstico temporal.
- `fecha_inicio`: comienzo de validez del tique; columna principal para ubicar el inicio del estacionamiento.
- `fecha_fin`: fin de validez del tique; columna principal para ubicar el final del estacionamiento.
- `cod_distrito`: código de distrito de validez del tique.
- `distrito`: nombre textual del distrito; no es clave crítica si `cod_distrito` está completo.
- `cod_barrio`: código de barrio de validez del tique; clave espacial principal.
- `barrio`: nombre textual del barrio.
- `tipo_zona`: tipo de plaza o ámbito SER.
- `distintivo`: distintivo ambiental del vehículo; se usa como diagnóstico, especialmente para interpretar comerciales y talleres.
- `minutos_tique`: duración declarada por el sistema; se contrasta con la duración calculada desde fechas.
- `importe_tique`: importe del tique.

#### Columnas descartadas del clean final

La salida final limpia no conservará `fecha_operacion`, `distintivo` ni `minutos_tique` como variables de modelado:

- `fecha_operacion` no representa el intervalo de validez del estacionamiento, sino el momento de obtención del tique.
- `distintivo` no es necesario para el objetivo principal de modelar dificultad SER a partir de eventos de tique; se usa previamente para diagnosticar comerciales/talleres.
- `minutos_tique` se sustituye por una duración derivada de `fecha_inicio` y `fecha_fin`, una vez superadas las validaciones temporales básicas.

La salida final conservará una duración calculada (`duracion_minutos`) y las claves temporales y espaciales necesarias para agregaciones posteriores. Las validaciones con calendario, parquímetros, calles y geometría se realizarán en un notebook posterior de joins SER.


In [71]:
SER_TIQUES_BASE_COLUMNS = [
    "matricula_parquimetro",
    "fecha_operacion",
    "fecha_inicio",
    "fecha_fin",
    "cod_distrito",
    "distrito",
    "cod_barrio",
    "barrio",
    "tipo_zona",
    "distintivo",
    "minutos_tique",
    "importe_tique",
]

SER_TIQUES_TEXT_COLUMNS = [
    "matricula_parquimetro",
    "distrito",
    "barrio",
    "tipo_zona",
    "distintivo",
]

SER_TIQUES_CODE_COLUMNS = [
    "cod_distrito",
    "cod_barrio",
]

SER_TIQUES_DATETIME_COLUMNS = [
    "fecha_operacion",
    "fecha_inicio",
    "fecha_fin",
]

SER_TIQUES_INTEGER_COLUMNS = [
    "minutos_tique",
]

SER_TIQUES_FLOAT_COLUMNS = [
    "importe_tique",
]

SER_TIQUES_FINAL_COLUMNS = [
    "matricula_parquimetro",
    "fecha_inicio",
    "fecha_fin",
    "duracion_minutos",
    "cod_distrito",
    "distrito",
    "cod_barrio",
    "barrio",
    "tipo_zona",
    "importe_tique",
]

print("Columnas base:", len(SER_TIQUES_BASE_COLUMNS))
print("Columnas finales previstas:", len(SER_TIQUES_FINAL_COLUMNS))
print("Columnas datetime:", len(SER_TIQUES_DATETIME_COLUMNS))
print("Columnas código:", len(SER_TIQUES_CODE_COLUMNS))
print("Columnas texto:", len(SER_TIQUES_TEXT_COLUMNS))


Columnas base: 12
Columnas finales previstas: 10
Columnas datetime: 3
Columnas código: 2
Columnas texto: 5


### 7.3. Fuentes efectivas y lectura incremental

La lectura se construye a partir de `ser_tiques_schema_summary`, que identifica qué periodos se leen directamente desde ZIP y cuál procede del CSV extraído manualmente del `.rar`.

Este bloque conserva la posibilidad de reconstruir la salida base desde raw, pero el notebook dará prioridad a los Parquet ya generados si existen, para evitar repetir una lectura completa innecesaria.


In [72]:
def infer_separator_from_header_bytes(data: bytes) -> str:
    first_line = data.splitlines()[0].decode("utf-8-sig", errors="replace")
    candidates = [";", ",", "\t", "|"]
    counts = {sep: first_line.count(sep) for sep in candidates}
    best_sep = max(counts, key=counts.get)

    if counts[best_sep] == 0:
        raise ValueError(f"No se pudo inferir separador en cabecera: {first_line[:200]}")

    return best_sep


def get_zip_member_header_bytes(zip_path: Path, member_name: str, n_bytes: int = 100_000) -> bytes:
    with zipfile.ZipFile(zip_path) as z:
        with z.open(member_name) as f:
            return f.read(n_bytes)


def get_file_header_bytes(path: Path, n_bytes: int = 100_000) -> bytes:
    with open(path, "rb") as f:
        return f.read(n_bytes)


def build_ser_tiques_sources(schema_summary: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for _, row in schema_summary.iterrows():
        if not bool(row["read_ok"]):
            raise ValueError(f"Periodo no legible: {row.to_dict()}")

        effective_file = Path(row["effective_file"])

        if row["source_type"] == "zip_internal_csv":
            physical_path = ROOT / row["zip_file"]
            member_name = row["member_name"]
            header_bytes = get_zip_member_header_bytes(physical_path, member_name)
        elif row["source_type"] == "manual_extracted_from_rar":
            physical_path = ROOT / effective_file
            member_name = None
            header_bytes = get_file_header_bytes(physical_path)
        else:
            raise ValueError(f"source_type no soportado: {row['source_type']}")

        sep = infer_separator_from_header_bytes(header_bytes)

        rows.append({
            "periodo_raw": row["periodo_raw"],
            "source_type": row["source_type"],
            "physical_path": str(physical_path.relative_to(ROOT)),
            "zip_file": row["zip_file"],
            "member_name": member_name,
            "separator": sep,
            "encoding": row["encoding_used"] if pd.notna(row["encoding_used"]) else "utf-8-sig",
            "n_columns": int(row["n_columns"]),
        })

    sources = pd.DataFrame(rows).sort_values("periodo_raw").reset_index(drop=True)

    if len(sources) != len(schema_summary):
        raise ValueError(f"Fuentes efectivas incompletas: {len(sources)} / {len(schema_summary)}")

    return sources


def iter_ser_tiques_raw_chunks(sources: pd.DataFrame, chunksize: int = 250_000):
    for _, source in sources.iterrows():
        physical_path = ROOT / source["physical_path"]

        read_kwargs = {
            "sep": source["separator"],
            "encoding": source["encoding"],
            "chunksize": chunksize,
            "low_memory": False,
        }

        if source["source_type"] == "zip_internal_csv":
            with zipfile.ZipFile(physical_path) as z:
                with z.open(source["member_name"]) as f:
                    for chunk in pd.read_csv(f, **read_kwargs):
                        yield source["periodo_raw"], source, chunk

        elif source["source_type"] == "manual_extracted_from_rar":
            for chunk in pd.read_csv(physical_path, **read_kwargs):
                yield source["periodo_raw"], source, chunk

        else:
            raise ValueError(f"source_type no soportado: {source['source_type']}")


ser_tiques_sources = build_ser_tiques_sources(ser_tiques_schema_summary)

sources_path = REPORTS_TABLES / "ser_tiques_sources_manifest.csv"
ser_tiques_sources.to_csv(sources_path, index=False)

display(ser_tiques_sources)


,periodo_raw,source_type,physical_path,zip_file,member_name,separator,encoding,n_columns
0,2023_q1,zip_internal_csv,data/raw/ser/ser_tiques/ser_tiques__2023_q1.zip,data/raw/ser/ser_tiques/ser_tiques__2023_q1.zip,1er 2023 TEMPO+NP.csv,;,utf-8-sig,12
1,2023_q2,zip_internal_csv,data/raw/ser/ser_tiques/ser_tiques__2023_q2.zip,data/raw/ser/ser_tiques/ser_tiques__2023_q2.zip,2otrimestre.csv,;,utf-8-sig,12
2,2023_q3,zip_internal_csv,data/raw/ser/ser_tiques/ser_tiques__2023_q3.zip,data/raw/ser/ser_tiques/ser_tiques__2023_q3.zip,tercertrimestre2023.csv,;,utf-8-sig,12
3,2023_q4,manual_extracted_from_rar,data/raw/ser/ser_tiques/_extracted_manual/ser_tiques__2023_q4.csv,data/raw/ser/ser_tiques/ser_tiques__2023_q4.zip,NaN,;,utf-8-sig,12
4,2024_q1,zip_internal_csv,data/raw/ser/ser_tiques/ser_tiques__2024_q1.zip,data/raw/ser/ser_tiques/ser_tiques__2024_q1.zip,primertrimestre2024.csv,",",utf-8-sig,12
5,2024_q2,zip_internal_csv,data/raw/ser/ser_tiques/ser_tiques__2024_q2.zip,data/raw/ser/ser_tiques/ser_tiques__2024_q2.zip,segundotrimestre2024.csv,",",utf-8-sig,12
6,2024_q3,zip_internal_csv,data/raw/ser/ser_tiques/ser_tiques__2024_q3.zip,data/raw/ser/ser_tiques/ser_tiques__2024_q3.zip,tercertrimestre2024.csv,",",utf-8-sig,12
7,2024_q4,zip_internal_csv,data/raw/ser/ser_tiques/ser_tiques__2024_q4.zip,data/raw/ser/ser_tiques/ser_tiques__2024_q4.zip,cuartotrimestre.csv,",",utf-8-sig,12
8,2025_q1,zip_internal_csv,data/raw/ser/ser_tiques/ser_tiques__2025_q1.zip,data/raw/ser/ser_tiques/ser_tiques__2025_q1.zip,Primertrimestre2025.csv,;,utf-8-sig,12
9,2025_q2,zip_internal_csv,data/raw/ser/ser_tiques/ser_tiques__2025_q2.zip,data/raw/ser/ser_tiques/ser_tiques__2025_q2.zip,Segundotrimestre2025.csv,;,utf-8-sig,12


### 7.4. Función de limpieza por chunk

La función `clean_ser_tiques_chunk()` aplica una limpieza base mínima:

- normaliza nombres de columnas;
- conserva únicamente las 12 columnas operativas;
- limpia espacios y valores vacíos;
- convierte fechas a `datetime`;
- convierte códigos territoriales a enteros nullable;
- convierte `minutos_tique` e `importe_tique` a formato numérico.

No descarta registros, no calcula duración derivada y no crea flags. Las reglas de exclusión se aplican más adelante sobre la salida base.


In [73]:
def clean_text_series(s: pd.Series) -> pd.Series:
    return (
        s.astype("string")
        .str.strip()
        .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA, "NULL": pd.NA})
    )


def parse_numeric_series(s: pd.Series) -> pd.Series:
    return pd.to_numeric(
        s.astype("string").str.replace(",", ".", regex=False),
        errors="coerce"
    )


def to_nullable_int_from_series(s: pd.Series) -> pd.Series:
    num = parse_numeric_series(s)
    non_null = num.dropna()
    if len(non_null) == 0:
        return num.astype("Int64")
    if ((non_null % 1) == 0).all():
        return num.astype("Int64")
    return num.astype("Float64")


def clean_ser_tiques_chunk(chunk_raw: pd.DataFrame, periodo_raw: str | None = None, source_row: pd.Series | None = None) -> pd.DataFrame:
    chunk = chunk_raw.copy()

    chunk.columns = [canonicalize_ser_tiques_column(c) for c in chunk.columns]

    missing = set(SER_TIQUES_REQUIRED_COLUMNS) - set(chunk.columns)
    if missing:
        raise ValueError(f"Faltan columnas requeridas en {periodo_raw}: {sorted(missing)}")

    chunk = chunk[SER_TIQUES_REQUIRED_COLUMNS].copy()

    for col in SER_TIQUES_TEXT_COLUMNS:
        chunk[col] = clean_text_series(chunk[col])

    for col in SER_TIQUES_CODE_COLUMNS:
        chunk[col] = to_nullable_int_from_series(chunk[col])

    for col in SER_TIQUES_DATETIME_COLUMNS:
        chunk[col] = pd.to_datetime(chunk[col], errors="coerce")

    chunk["minutos_tique"] = parse_numeric_series(chunk["minutos_tique"]).round().astype("Int64")
    chunk["importe_tique"] = parse_numeric_series(chunk["importe_tique"]).astype("float64")

    return chunk[SER_TIQUES_BASE_COLUMNS].copy()


clean_smoke_rows = []

for _, source in ser_tiques_sources.iterrows():
    periodo_raw, source_row, chunk_raw = next(iter_ser_tiques_raw_chunks(
        sources=pd.DataFrame([source]),
        chunksize=1_000
    ))

    chunk_clean = clean_ser_tiques_chunk(chunk_raw, periodo_raw, source_row)

    clean_smoke_rows.append({
        "periodo_raw": periodo_raw,
        "rows_sample": len(chunk_clean),
        "n_columns_clean": chunk_clean.shape[1],
        "columns_ok": list(chunk_clean.columns) == SER_TIQUES_BASE_COLUMNS,
        "n_fecha_operacion_null": int(chunk_clean["fecha_operacion"].isna().sum()),
        "n_fecha_inicio_null": int(chunk_clean["fecha_inicio"].isna().sum()),
        "n_fecha_fin_null": int(chunk_clean["fecha_fin"].isna().sum()),
        "n_minutos_null": int(chunk_clean["minutos_tique"].isna().sum()),
        "n_importe_null": int(chunk_clean["importe_tique"].isna().sum()),
    })

ser_tiques_clean_smoke_test = pd.DataFrame(clean_smoke_rows)
ser_tiques_clean_smoke_test.to_csv(REPORTS_TABLES / "ser_tiques_clean_smoke_test.csv", index=False)

display(ser_tiques_clean_smoke_test)

if (
    ser_tiques_clean_smoke_test["columns_ok"].all()
    and ser_tiques_clean_smoke_test["n_columns_clean"].eq(len(SER_TIQUES_BASE_COLUMNS)).all()
):
    print("Resultado: la función de limpieza base devuelve 12 columnas operativas en todos los periodos.")
else:
    raise RuntimeError("Algún periodo no devuelve el esquema limpio base esperado.")


,periodo_raw,rows_sample,n_columns_clean,columns_ok,n_fecha_operacion_null,n_fecha_inicio_null,n_fecha_fin_null,n_minutos_null,n_importe_null
0,2023_q1,1000,12,True,0,0,0,0,0
1,2023_q2,1000,12,True,0,0,0,0,0
2,2023_q3,1000,12,True,0,0,0,0,0
3,2023_q4,1000,12,True,0,0,0,0,0
4,2024_q1,1000,12,True,0,0,0,0,0
5,2024_q2,1000,12,True,0,0,0,0,0
6,2024_q3,1000,12,True,0,0,0,0,0
7,2024_q4,1000,12,True,0,0,0,0,0
8,2025_q1,1000,12,True,0,0,0,0,0
9,2025_q2,1000,12,True,0,0,0,0,0


Resultado: la función de limpieza base devuelve 12 columnas operativas en todos los periodos.


### 7.5. Preparación de Parquet base sin releer raw si ya existen partes limpias

Para evitar repetir la lectura completa de los ZIP, el notebook prioriza los Parquet ya generados en `data/interim/ser/ser_tiques/clean_parts/`.

Si existen partes limpias previas con columnas de trazabilidad añadidas, se usan como staging y se proyectan a una nueva salida base con solo 12 columnas operativas:

`data/interim/ser/ser_tiques/base_clean_parts/`

Si no existen partes previas, se reconstruyen desde raw con la función de limpieza por chunks.

Esta salida base sigue particionada por `periodo_raw` en la ruta de carpetas, pero `periodo_raw` no queda como columna física dentro del Parquet.


In [74]:
import shutil
import time
from glob import glob

try:
    import pyarrow
    PARQUET_ENGINE = "pyarrow"
except ImportError as exc:
    raise ImportError(
        "Falta pyarrow para escribir Parquet. Instálalo antes de continuar: "
        "conda install -c conda-forge pyarrow"
    ) from exc

LEGACY_CLEAN_PARTS_DIR = ROOT / "data/interim/ser/ser_tiques/clean_parts"
BASE_CLEAN_PARTS_DIR = ROOT / "data/interim/ser/ser_tiques/base_clean_parts"

BASE_MANIFEST_PATH = REPORTS_TABLES / "ser_tiques_base_clean_parts_manifest.csv"
LEGACY_MANIFEST_PATH = REPORTS_TABLES / "ser_tiques_clean_parts_manifest.csv"

CLEANING_CHUNKSIZE = 250_000
RECREATE_BASE_PARTS = False


def extract_periodo_from_partition_path(path: Path) -> str:
    for part in path.parts:
        if part.startswith("periodo_raw="):
            return part.split("=", 1)[1]
    raise ValueError(f"No se puede extraer periodo_raw de la ruta: {path}")


def parquet_files_under(path: Path) -> list[Path]:
    return sorted(Path(p) for p in glob(str(path / "**/*.parquet"), recursive=True))


base_files_existing = parquet_files_under(BASE_CLEAN_PARTS_DIR)

if RECREATE_BASE_PARTS and BASE_CLEAN_PARTS_DIR.exists():
    shutil.rmtree(BASE_CLEAN_PARTS_DIR)
    base_files_existing = []

manifest_rows = []
t0 = time.time()

if base_files_existing and BASE_MANIFEST_PATH.exists():
    print("Se reutiliza la salida base existente:", BASE_CLEAN_PARTS_DIR.relative_to(ROOT))
    ser_tiques_base_parts_manifest = pd.read_csv(BASE_MANIFEST_PATH)

elif LEGACY_CLEAN_PARTS_DIR.exists() and parquet_files_under(LEGACY_CLEAN_PARTS_DIR):
    print("Se proyectan Parquet existentes a salida base de 12 columnas.")

    if BASE_CLEAN_PARTS_DIR.exists():
        shutil.rmtree(BASE_CLEAN_PARTS_DIR)

    BASE_CLEAN_PARTS_DIR.mkdir(parents=True, exist_ok=True)

    legacy_files = parquet_files_under(LEGACY_CLEAN_PARTS_DIR)

    for i, legacy_path in enumerate(legacy_files):
        periodo_raw = extract_periodo_from_partition_path(legacy_path)
        df = pd.read_parquet(legacy_path, engine=PARQUET_ENGINE)

        df.columns = [canonicalize_ser_tiques_column(c) for c in df.columns]
        df = df[SER_TIQUES_BASE_COLUMNS].copy()

        for col in SER_TIQUES_TEXT_COLUMNS:
            df[col] = clean_text_series(df[col])

        for col in SER_TIQUES_CODE_COLUMNS:
            df[col] = to_nullable_int_from_series(df[col])

        for col in SER_TIQUES_DATETIME_COLUMNS:
            df[col] = pd.to_datetime(df[col], errors="coerce")

        df["minutos_tique"] = parse_numeric_series(df["minutos_tique"]).round().astype("Int64")
        df["importe_tique"] = parse_numeric_series(df["importe_tique"]).astype("float64")

        period_dir = BASE_CLEAN_PARTS_DIR / f"periodo_raw={periodo_raw}"
        period_dir.mkdir(parents=True, exist_ok=True)

        output_path = period_dir / legacy_path.name

        df.to_parquet(output_path, engine=PARQUET_ENGINE, compression="snappy", index=False)

        manifest_rows.append({
            "periodo_raw": periodo_raw,
            "part_number": i,
            "n_rows": len(df),
            "n_columns": df.shape[1],
            "output_path": str(output_path.relative_to(ROOT)),
            "output_size_mb": round(output_path.stat().st_size / (1024 ** 2), 3),
            "source": "projected_from_existing_clean_parts",
        })

        if (i + 1) % 50 == 0:
            print(f"Partes proyectadas: {i + 1:,} / {len(legacy_files):,}")

    ser_tiques_base_parts_manifest = pd.DataFrame(manifest_rows)
    ser_tiques_base_parts_manifest.to_csv(BASE_MANIFEST_PATH, index=False)

else:
    print("No existen Parquet previos. Se reconstruye salida base desde raw.")

    if BASE_CLEAN_PARTS_DIR.exists():
        shutil.rmtree(BASE_CLEAN_PARTS_DIR)

    BASE_CLEAN_PARTS_DIR.mkdir(parents=True, exist_ok=True)

    part_counter_by_period = {}
    total_rows = 0

    for periodo_raw, source_row, chunk_raw in iter_ser_tiques_raw_chunks(
        ser_tiques_sources,
        chunksize=CLEANING_CHUNKSIZE
    ):
        chunk_clean = clean_ser_tiques_chunk(chunk_raw, periodo_raw, source_row)

        part_number = part_counter_by_period.get(periodo_raw, 0)
        part_counter_by_period[periodo_raw] = part_number + 1

        period_dir = BASE_CLEAN_PARTS_DIR / f"periodo_raw={periodo_raw}"
        period_dir.mkdir(parents=True, exist_ok=True)

        output_path = period_dir / f"part_{part_number:05d}.parquet"

        chunk_clean.to_parquet(
            output_path,
            engine=PARQUET_ENGINE,
            compression="snappy",
            index=False
        )

        n_rows = len(chunk_clean)
        total_rows += n_rows

        manifest_rows.append({
            "periodo_raw": periodo_raw,
            "part_number": part_number,
            "n_rows": n_rows,
            "n_columns": chunk_clean.shape[1],
            "output_path": str(output_path.relative_to(ROOT)),
            "output_size_mb": round(output_path.stat().st_size / (1024 ** 2), 3),
            "source": "rebuilt_from_raw",
        })

        if len(manifest_rows) % 10 == 0:
            elapsed_min = (time.time() - t0) / 60
            print(
                f"Partes escritas: {len(manifest_rows)} | "
                f"Filas procesadas: {total_rows:,} | "
                f"Tiempo: {elapsed_min:.1f} min"
            )

    ser_tiques_base_parts_manifest = pd.DataFrame(manifest_rows)
    ser_tiques_base_parts_manifest.to_csv(BASE_MANIFEST_PATH, index=False)

elapsed_min = (time.time() - t0) / 60

print("Preparación de Parquet base finalizada.")
print("Partes base:", len(ser_tiques_base_parts_manifest))
print("Filas base:", f"{int(ser_tiques_base_parts_manifest['n_rows'].sum()):,}")
print("Columnas base por parte:", sorted(ser_tiques_base_parts_manifest["n_columns"].unique().tolist()))
print(f"Tiempo total: {elapsed_min:.1f} min")

summary_by_period = (
    ser_tiques_base_parts_manifest
    .groupby("periodo_raw")
    .agg(
        n_parts=("part_number", "count"),
        n_rows=("n_rows", "sum"),
        total_size_mb=("output_size_mb", "sum"),
    )
    .reset_index()
)

display(summary_by_period)

sample_output = ROOT / ser_tiques_base_parts_manifest.iloc[0]["output_path"]
sample_back = pd.read_parquet(sample_output, engine=PARQUET_ENGINE)

print("Muestra re-leída:", sample_output.relative_to(ROOT))
print("Shape muestra:", sample_back.shape)
print("Columnas correctas:", list(sample_back.columns) == SER_TIQUES_BASE_COLUMNS)


Se reutiliza la salida base existente: data/interim/ser/ser_tiques/base_clean_parts
Preparación de Parquet base finalizada.
Partes base: 606
Filas base: 150,139,924
Columnas base por parte: [12]
Tiempo total: 0.0 min


,periodo_raw,n_parts,n_rows,total_size_mb
0,2023_q1,48,11908878,336.272
1,2023_q2,47,11665770,271.448
2,2023_q3,39,9651702,278.855
3,2023_q4,47,11561992,335.170
4,2024_q1,49,12104826,354.171
5,2024_q2,50,12490979,365.530
6,2024_q3,32,7907706,214.115
7,2024_q4,51,12597460,369.373
8,2025_q1,51,12745646,349.023
9,2025_q2,50,12361312,349.142


Muestra re-leída: data/interim/ser/ser_tiques/base_clean_parts/periodo_raw=2023_q1/part_00000.parquet
Shape muestra: (250000, 12)
Columnas correctas: True


### 7.6. Validaciones intrínsecas de la salida base

Las validaciones se realizan sobre `base_clean_parts`, sin cruzar todavía con calendario, parquímetros, calles ni geometría.

El objetivo es cerrar la limpieza individual de `ser_tiques` como fuente base utilizable. Las validaciones cruzadas del régimen SER, la ubicación de parquímetros, la coherencia con calles/plazas y la cobertura cartográfica se moverán a un notebook posterior de joins SER.


In [75]:
import duckdb

PARQUET_GLOB = str((BASE_CLEAN_PARTS_DIR / "**/*.parquet").as_posix())
BASE_MANIFEST_PATH = REPORTS_TABLES / "ser_tiques_base_clean_parts_manifest.csv"

RUN_DUPLICATE_CHECK = True

con = duckdb.connect()

print("DuckDB:", duckdb.__version__)
print("Parquet glob:", PARQUET_GLOB)
print("Manifest base existe:", BASE_MANIFEST_PATH.exists())

if not BASE_MANIFEST_PATH.exists():
    raise FileNotFoundError(BASE_MANIFEST_PATH)

manifest = pd.read_csv(BASE_MANIFEST_PATH)

n_manifest_parts = len(manifest)
n_manifest_rows = int(manifest["n_rows"].sum())

print("Partes en manifest:", n_manifest_parts)
print("Filas en manifest:", f"{n_manifest_rows:,}")


DuckDB: 1.5.3
Parquet glob: /Users/hugo/TFM_parking_madrid/data/interim/ser/ser_tiques/base_clean_parts/**/*.parquet
Manifest base existe: True
Partes en manifest: 606
Filas en manifest: 150,139,924


### 7.7. Integridad de escritura

Se comprueba que el número de filas en Parquet coincide con el manifiesto y que el esquema físico contiene exactamente las 12 columnas base.


In [76]:
n_parquet_rows = con.execute(f'''
    SELECT COUNT(*) AS n_rows
    FROM read_parquet('{PARQUET_GLOB}', hive_partitioning=true)
''').fetchone()[0]

sample_schema = con.execute(f'''
    SELECT *
    FROM read_parquet('{PARQUET_GLOB}', hive_partitioning=true)
    LIMIT 0
''').df()

actual_columns = [c for c in sample_schema.columns if c != "periodo_raw"]
expected_columns = SER_TIQUES_BASE_COLUMNS

validation_overview = pd.DataFrame([
    {
        "check": "n_partes_manifest",
        "valor": n_manifest_parts,
        "esperado": ">= 1",
        "ok": n_manifest_parts >= 1,
        "lectura": "Número de partes Parquet registradas en el manifiesto.",
    },
    {
        "check": "n_filas_manifest_vs_parquet",
        "valor": n_parquet_rows,
        "esperado": n_manifest_rows,
        "ok": n_parquet_rows == n_manifest_rows,
        "lectura": "Las filas escritas en Parquet coinciden con el manifiesto.",
    },
    {
        "check": "n_columnas_base",
        "valor": len(actual_columns),
        "esperado": len(expected_columns),
        "ok": actual_columns == expected_columns,
        "lectura": "El contenido físico de los Parquet mantiene las 12 columnas operativas.",
    },
])

display(validation_overview)


,check,valor,esperado,ok,lectura
0,n_partes_manifest,606,>= 1,True,Número de partes Parquet registradas en el manifiesto.
1,n_filas_manifest_vs_parquet,150139924,150139924,True,Las filas escritas en Parquet coinciden con el manifiesto.
2,n_columnas_base,12,12,True,El contenido físico de los Parquet mantiene las 12 columnas operativas.


**Lectura.**

La integridad de escritura se considera válida si el manifiesto y la lectura Parquet devuelven el mismo número de filas y si el esquema físico contiene únicamente las 12 columnas base. Esta comprobación no valida aún la calidad de los tiques, pero sí cierra la parte técnica de escritura y lectura como dataset lógico.


### 7.8. Validaciones temporales y cobertura trimestral

Se construyen los límites esperados de cada periodo y se deriva un `periodo_inicio` a partir de `fecha_inicio`.

La distinción entre `periodo_raw` y `periodo_inicio` es importante. `periodo_raw` identifica el archivo trimestral de origen; `periodo_inicio` identifica el trimestre analítico real del tique según su fecha de inicio. Por tanto, los registros cuya `fecha_inicio` cae fuera del trimestre del archivo no se eliminan automáticamente: si pertenecen a otro trimestre cubierto por la fuente, se conservan y se reasignan al periodo analítico correspondiente.

En esta fase se cuantifican:

- fechas nulas;
- `fecha_fin < fecha_inicio`;
- registros fuera de la ventana global cubierta por la fuente;
- registros reasignables a otro periodo cubierto;
- intervalos cuya `fecha_fin` cruza de trimestre o sale de la ventana global;
- cobertura mensual global.

La duración cero no se evalúa aquí, porque la variable final de duración se deriva explícitamente en la sección siguiente.


In [77]:
period_bounds = []

for periodo in sorted(manifest["periodo_raw"].unique()):
    year_str, quarter_str = periodo.split("_q")
    year = int(year_str)
    quarter = int(quarter_str)

    start_month = (quarter - 1) * 3 + 1
    start_date = pd.Timestamp(year=year, month=start_month, day=1)

    if quarter < 4:
        end_date_exclusive = pd.Timestamp(year=year, month=start_month + 3, day=1)
    else:
        end_date_exclusive = pd.Timestamp(year=year + 1, month=1, day=1)

    period_bounds.append({
        "periodo_raw": periodo,
        "period_start": start_date.date(),
        "period_end_exclusive": end_date_exclusive.date(),
    })

period_bounds_df = pd.DataFrame(period_bounds)
con.register("period_bounds", period_bounds_df)

MIN_GLOBAL_DATE = "2023-01-01"
MAX_GLOBAL_DATE_EXCLUSIVE = "2026-04-01"

con.execute(f'''
    CREATE OR REPLACE VIEW ser_tiques_with_period AS
    SELECT
        t.*,
        b.period_start,
        b.period_end_exclusive,
        CASE
            WHEN t.fecha_inicio >= TIMESTAMP '{MIN_GLOBAL_DATE}'
             AND t.fecha_inicio < TIMESTAMP '{MAX_GLOBAL_DATE_EXCLUSIVE}'
            THEN CAST(EXTRACT(year FROM t.fecha_inicio) AS VARCHAR)
                 || '_q'
                 || CAST(EXTRACT(quarter FROM t.fecha_inicio) AS VARCHAR)
            ELSE NULL
        END AS periodo_inicio,
        CASE
            WHEN t.fecha_fin >= TIMESTAMP '{MIN_GLOBAL_DATE}'
             AND t.fecha_fin < TIMESTAMP '{MAX_GLOBAL_DATE_EXCLUSIVE}'
            THEN CAST(EXTRACT(year FROM t.fecha_fin) AS VARCHAR)
                 || '_q'
                 || CAST(EXTRACT(quarter FROM t.fecha_fin) AS VARCHAR)
            ELSE NULL
        END AS periodo_fin,
        date_diff('second', t.fecha_inicio, t.fecha_fin) AS duracion_segundos
    FROM read_parquet('{PARQUET_GLOB}', hive_partitioning=true) AS t
    LEFT JOIN period_bounds AS b
        ON t.periodo_raw = b.periodo_raw
''')

temporal_checks_raw = con.execute(f'''
    WITH month_coverage AS (
        SELECT
            COUNT(DISTINCT date_trunc('month', fecha_inicio)) AS n_meses_con_tiques
        FROM ser_tiques_with_period
        WHERE fecha_inicio >= TIMESTAMP '{MIN_GLOBAL_DATE}'
          AND fecha_inicio < TIMESTAMP '{MAX_GLOBAL_DATE_EXCLUSIVE}'
    )
    SELECT
        COUNT(*) AS n_rows,
        MIN(fecha_inicio) AS min_fecha_inicio,
        MAX(fecha_inicio) AS max_fecha_inicio,
        SUM(CASE WHEN fecha_operacion IS NULL THEN 1 ELSE 0 END) AS n_fecha_operacion_null,
        SUM(CASE WHEN fecha_inicio IS NULL THEN 1 ELSE 0 END) AS n_fecha_inicio_null,
        SUM(CASE WHEN fecha_fin IS NULL THEN 1 ELSE 0 END) AS n_fecha_fin_null,
        SUM(CASE WHEN fecha_inicio < TIMESTAMP '{MIN_GLOBAL_DATE}' THEN 1 ELSE 0 END) AS n_fecha_inicio_antes_2023,
        SUM(CASE WHEN fecha_inicio >= TIMESTAMP '{MAX_GLOBAL_DATE_EXCLUSIVE}' THEN 1 ELSE 0 END) AS n_fecha_inicio_desde_2026_04_01,
        SUM(CASE WHEN periodo_inicio IS NULL THEN 1 ELSE 0 END) AS n_fecha_inicio_fuera_ventana_global,
        SUM(CASE WHEN fecha_fin < fecha_inicio THEN 1 ELSE 0 END) AS n_fecha_fin_menor_inicio,
        SUM(CASE WHEN fecha_fin < TIMESTAMP '{MIN_GLOBAL_DATE}'
                  OR fecha_fin >= TIMESTAMP '{MAX_GLOBAL_DATE_EXCLUSIVE}'
                 THEN 1 ELSE 0 END) AS n_fecha_fin_fuera_ventana_global,
        SUM(CASE
            WHEN fecha_inicio IS NOT NULL
             AND periodo_inicio IS NOT NULL
             AND periodo_inicio <> periodo_raw
            THEN 1 ELSE 0
        END) AS n_fecha_inicio_reasignable_a_otro_periodo,
        SUM(CASE
            WHEN periodo_inicio IS NOT NULL
             AND periodo_fin IS NOT NULL
             AND periodo_fin <> periodo_inicio
            THEN 1 ELSE 0
        END) AS n_intervalo_cruza_trimestre_inicio,
        (SELECT n_meses_con_tiques FROM month_coverage) AS n_meses_con_tiques_2023_01_a_2026_03
    FROM ser_tiques_with_period
''').df()

temporal_row = temporal_checks_raw.iloc[0].to_dict()

temporal_checks = pd.DataFrame([
    {
        "validacion": "fecha_operacion_nula",
        "n_afectadas": int(temporal_row["n_fecha_operacion_null"]),
        "decision": "diagnostico",
        "accion": "no se conserva en salida final",
    },
    {
        "validacion": "fecha_inicio_nula",
        "n_afectadas": int(temporal_row["n_fecha_inicio_null"]),
        "decision": "excluir si aparece",
        "accion": "requerida para intervalo temporal",
    },
    {
        "validacion": "fecha_fin_nula",
        "n_afectadas": int(temporal_row["n_fecha_fin_null"]),
        "decision": "excluir si aparece",
        "accion": "requerida para intervalo temporal",
    },
    {
        "validacion": "fecha_fin_menor_inicio",
        "n_afectadas": int(temporal_row["n_fecha_fin_menor_inicio"]),
        "decision": "excluir",
        "accion": "no representa intervalo temporal válido",
    },
    {
        "validacion": "fecha_inicio_fuera_ventana_global",
        "n_afectadas": int(temporal_row["n_fecha_inicio_fuera_ventana_global"]),
        "decision": "excluir",
        "accion": "queda fuera de la cobertura temporal del dataset",
    },
    {
        "validacion": "fecha_inicio_antes_2023",
        "n_afectadas": int(temporal_row["n_fecha_inicio_antes_2023"]),
        "decision": "excluir",
        "accion": "fuera de la ventana global cubierta",
    },
    {
        "validacion": "fecha_inicio_desde_2026_04_01",
        "n_afectadas": int(temporal_row["n_fecha_inicio_desde_2026_04_01"]),
        "decision": "excluir",
        "accion": "pertenece a un trimestre no cubierto por la fuente actual",
    },
    {
        "validacion": "fecha_inicio_reasignable_a_otro_periodo",
        "n_afectadas": int(temporal_row["n_fecha_inicio_reasignable_a_otro_periodo"]),
        "decision": "conservar_reasignando",
        "accion": "se conserva usando periodo_inicio como partición analítica",
    },
    {
        "validacion": "fecha_fin_fuera_ventana_global",
        "n_afectadas": int(temporal_row["n_fecha_fin_fuera_ventana_global"]),
        "decision": "diagnostico",
        "accion": "no se filtra si fecha_inicio pertenece a la ventana cubierta",
    },
    {
        "validacion": "intervalo_cruza_trimestre_inicio",
        "n_afectadas": int(temporal_row["n_intervalo_cruza_trimestre_inicio"]),
        "decision": "diagnostico",
        "accion": "puede requerir reparto temporal en panel horario posterior",
    },
    {
        "validacion": "cobertura_mensual_2023_01_a_2026_03",
        "n_afectadas": int(temporal_row["n_meses_con_tiques_2023_01_a_2026_03"]),
        "decision": "ok si 39",
        "accion": "confirma cobertura mensual esperada",
    },
])

temporal_checks["pct_afectado"] = (
    temporal_checks["n_afectadas"] / int(temporal_row["n_rows"]) * 100
).round(6)

display(period_bounds_df)
display(temporal_checks)

con.execute('''
    CREATE OR REPLACE VIEW ser_tiques_fechas_validas AS
    SELECT *
    FROM ser_tiques_with_period
    WHERE fecha_inicio IS NOT NULL
      AND fecha_fin IS NOT NULL
      AND fecha_fin >= fecha_inicio
      AND periodo_inicio IS NOT NULL
''')

n_fechas_validas = con.execute('''
    SELECT COUNT(*) FROM ser_tiques_fechas_validas
''').fetchone()[0]

print("Filas base:", f"{int(temporal_row['n_rows']):,}")
print("Filas con fechas válidas y dentro de ventana global:", f"{n_fechas_validas:,}")
print("Filas excluidas por validación temporal de fechas:", f"{int(temporal_row['n_rows']) - n_fechas_validas:,}")

temporal_checks.to_csv(REPORTS_TABLES / "ser_tiques_temporal_checks_global.csv", index=False)
period_bounds_df.to_csv(REPORTS_TABLES / "ser_tiques_period_bounds.csv", index=False)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,periodo_raw,period_start,period_end_exclusive
0,2023_q1,2023-01-01,2023-04-01
1,2023_q2,2023-04-01,2023-07-01
2,2023_q3,2023-07-01,2023-10-01
3,2023_q4,2023-10-01,2024-01-01
4,2024_q1,2024-01-01,2024-04-01
5,2024_q2,2024-04-01,2024-07-01
6,2024_q3,2024-07-01,2024-10-01
7,2024_q4,2024-10-01,2025-01-01
8,2025_q1,2025-01-01,2025-04-01
9,2025_q2,2025-04-01,2025-07-01


,validacion,n_afectadas,decision,accion,pct_afectado
0,fecha_operacion_nula,0,diagnostico,no se conserva en salida final,0.000000
1,fecha_inicio_nula,0,excluir si aparece,requerida para intervalo temporal,0.000000
2,fecha_fin_nula,0,excluir si aparece,requerida para intervalo temporal,0.000000
3,fecha_fin_menor_inicio,91,excluir,no representa intervalo temporal válido,0.000061
4,fecha_inicio_fuera_ventana_global,654,excluir,queda fuera de la cobertura temporal del dataset,0.000436
5,fecha_inicio_antes_2023,3,excluir,fuera de la ventana global cubierta,0.000002
6,fecha_inicio_desde_2026_04_01,651,excluir,pertenece a un trimestre no cubierto por la fuente actual,0.000434
7,fecha_inicio_reasignable_a_otro_periodo,23415,conservar_reasignando,se conserva usando periodo_inicio como partición analítica,0.015595
8,fecha_fin_fuera_ventana_global,9632,diagnostico,no se filtra si fecha_inicio pertenece a la ventana cubierta,0.006415
9,intervalo_cruza_trimestre_inicio,111741,diagnostico,puede requerir reparto temporal en panel horario posterior,0.074425


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Filas base: 150,139,924
Filas con fechas válidas y dentro de ventana global: 150,139,179
Filas excluidas por validación temporal de fechas: 745


**Lectura.**

La validación temporal de fechas no detecta nulos en las fechas principales, pero sí identifica registros con `fecha_fin < fecha_inicio` y registros fuera de la ventana global cubierta por la fuente.

Los registros cuya `fecha_inicio` no coincide con el trimestre del archivo de origen no se eliminan si pueden asignarse a otro trimestre cubierto por la fuente. En esos casos se conserva el registro y se utiliza `periodo_inicio` como partición analítica. Esto no implica reordenar físicamente los Parquet: el orden cronológico se impondrá en los paneles posteriores mediante ordenación o agregación por fecha, no por la posición física de las filas.

La `fecha_fin` se controla mediante dos diagnósticos adicionales: si queda fuera de la ventana global y si cruza a un trimestre distinto del inicio. No se exige que `fecha_fin` pertenezca al mismo trimestre que `fecha_inicio`, porque un intervalo puede empezar al final de un trimestre y terminar al comienzo del siguiente. Estos casos deberán tratarse con cuidado si el panel horario posterior reparte duración entre horas o recorta intervalos por ventana temporal.

A partir de este punto, el notebook trabaja secuencialmente sobre `ser_tiques_fechas_validas`, de modo que las comprobaciones de duración, claves, tipo de zona, límites normativos y duplicados no se realizan sobre registros que ya han quedado fuera por problemas temporales básicos.


### 7.9. Validación de duración: `minutos_tique` frente a duración calculada

Una vez filtradas las fechas temporalmente válidas, se deriva la duración desde `fecha_inicio` y `fecha_fin`.

La duración cero se evalúa aquí porque la variable final `duracion_minutos` procede de esas fechas. Los registros con duración nula no representan un intervalo positivo de estacionamiento y quedan fuera de la salida final.

Después se compara `minutos_tique` con la duración calculada mediante redondeo hacia arriba (`ceil(segundos/60)`). Esta comparación sirve para decidir qué variable de duración se conserva.


In [78]:
duration_zero_check = con.execute('''
    SELECT
        COUNT(*) AS n_rows_fechas_validas,
        SUM(CASE WHEN duracion_segundos = 0 THEN 1 ELSE 0 END) AS n_duracion_cero_calculada,
        SUM(CASE WHEN duracion_segundos > 0 THEN 1 ELSE 0 END) AS n_rows_validos_para_duracion
    FROM ser_tiques_fechas_validas
''').df()

display(duration_zero_check)

con.execute('''
    CREATE OR REPLACE VIEW ser_tiques_temporal_valid AS
    SELECT
        *,
        CAST(CEIL(duracion_segundos / 60.0) AS BIGINT) AS duracion_minutos
    FROM ser_tiques_fechas_validas
    WHERE duracion_segundos > 0
''')

duration_comparison = con.execute('''
    WITH calc AS (
        SELECT
            *,
            CAST(CEIL(duracion_segundos / 60.0) AS BIGINT) AS duracion_min_ceil
        FROM ser_tiques_fechas_validas
        WHERE duracion_segundos > 0
    )
    SELECT
        COUNT(*) AS n_rows_validos_para_duracion,
        SUM(CASE WHEN minutos_tique IS NULL THEN 1 ELSE 0 END) AS n_minutos_tique_null,
        SUM(CASE WHEN minutos_tique = duracion_min_ceil THEN 1 ELSE 0 END) AS n_coincide_ceil,
        SUM(CASE WHEN minutos_tique <> duracion_min_ceil THEN 1 ELSE 0 END) AS n_dif_ceil,
        SUM(CASE WHEN ABS(minutos_tique - duracion_min_ceil) <= 1 THEN 1 ELSE 0 END) AS n_dif_ceil_abs_le_1,
        SUM(CASE WHEN ABS(minutos_tique - duracion_min_ceil) > 120 THEN 1 ELSE 0 END) AS n_dif_ceil_abs_gt_120,
        MIN(minutos_tique - duracion_min_ceil) AS min_dif_ceil,
        MAX(minutos_tique - duracion_min_ceil) AS max_dif_ceil
    FROM calc
''').df()

duration_diff_distribution = con.execute('''
    WITH diffed AS (
        SELECT
            minutos_tique - duracion_minutos AS dif_minutos_ceil
        FROM ser_tiques_temporal_valid
    )
    SELECT
        CASE
            WHEN dif_minutos_ceil = 0 THEN '0'
            WHEN ABS(dif_minutos_ceil) = 1 THEN '+/-1'
            WHEN ABS(dif_minutos_ceil) BETWEEN 2 AND 5 THEN '+/-2_5'
            WHEN ABS(dif_minutos_ceil) BETWEEN 6 AND 30 THEN '+/-6_30'
            WHEN ABS(dif_minutos_ceil) BETWEEN 31 AND 120 THEN '+/-31_120'
            ELSE 'mayor_120'
        END AS tramo_diferencia,
        COUNT(*) AS n_rows
    FROM diffed
    GROUP BY tramo_diferencia
    ORDER BY
        CASE tramo_diferencia
            WHEN '0' THEN 0
            WHEN '+/-1' THEN 1
            WHEN '+/-2_5' THEN 2
            WHEN '+/-6_30' THEN 3
            WHEN '+/-31_120' THEN 4
            ELSE 5
        END
''').df()

duration_diff_distribution["pct_rows"] = (
    duration_diff_distribution["n_rows"]
    / int(duration_comparison["n_rows_validos_para_duracion"].iloc[0])
    * 100
).round(6)

display(duration_diff_distribution)

duration_zero_check.to_csv(REPORTS_TABLES / "ser_tiques_duration_zero_check.csv", index=False)
duration_comparison.to_csv(REPORTS_TABLES / "ser_tiques_duration_comparison.csv", index=False)
duration_diff_distribution.to_csv(REPORTS_TABLES / "ser_tiques_duration_diff_distribution.csv", index=False)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_rows_fechas_validas,n_duracion_cero_calculada,n_rows_validos_para_duracion
0,150139179,1288.0,150137891.0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,tramo_diferencia,n_rows,pct_rows
0,0,140525913,93.597900
1,+/-1,634413,0.422554
2,+/-2_5,119171,0.079374
3,+/-6_30,11584,0.007716
4,+/-31_120,66,0.000044
5,mayor_120,8846744,5.892413


**Lectura/decisión.**

Tras excluir los registros con fechas incoherentes o fuera de la ventana global cubierta, la duración final se calcula desde `fecha_inicio` y `fecha_fin`. Los registros con duración cero se excluyen porque no representan un intervalo positivo de estacionamiento.

`minutos_tique` se utiliza como contraste, pero no se conserva como variable final. La tabla por tramos muestra que la mayoría de registros coincide exactamente o presenta diferencias residuales, pero también existe un bloque relevante con discrepancias superiores a 120 minutos. Para el modelado espacio-temporal resulta más consistente usar una duración derivada directamente del intervalo de validez del tique.


### 7.10. Nulos en claves espaciales y matrícula de parquímetro

Esta validación se realiza sobre `ser_tiques_temporal_valid`, es decir, únicamente sobre registros que ya superaron las comprobaciones de fechas y duración positiva.

Los campos críticos son:

- `cod_distrito`;
- `cod_barrio`;
- `barrio`;
- `matricula_parquimetro`.

El campo `distrito` se considera descriptivo y reconstruible si `cod_distrito` está completo.


In [79]:
key_nulls_global = con.execute('''
    SELECT
        COUNT(*) AS n_rows,
        SUM(CASE WHEN cod_distrito IS NULL THEN 1 ELSE 0 END) AS n_cod_distrito_null,
        SUM(CASE WHEN distrito IS NULL THEN 1 ELSE 0 END) AS n_distrito_null,
        SUM(CASE WHEN cod_barrio IS NULL THEN 1 ELSE 0 END) AS n_cod_barrio_null,
        SUM(CASE WHEN barrio IS NULL THEN 1 ELSE 0 END) AS n_barrio_null,
        SUM(CASE WHEN matricula_parquimetro IS NULL THEN 1 ELSE 0 END) AS n_matricula_parquimetro_null
    FROM ser_tiques_temporal_valid
''').df()

display(key_nulls_global)

key_nulls_global.to_csv(REPORTS_TABLES / "ser_tiques_key_nulls_global.csv", index=False)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_rows,n_cod_distrito_null,n_distrito_null,n_cod_barrio_null,n_barrio_null,n_matricula_parquimetro_null
0,150137891,0.0,404272.0,0.0,0.0,0.0


**Lectura.**

La fuente se considera válida para agregación espacial si `cod_barrio`, `barrio` y `matricula_parquimetro` no presentan nulos tras el filtrado temporal. Los nulos en `distrito` textual no bloquean la limpieza porque el código de distrito puede actuar como clave y el nombre puede reconstruirse posteriormente mediante tablas de referencia.


### 7.11. Diagnóstico de `tipo_zona`, `distintivo` y comerciales/talleres

Esta comprobación se realiza sobre `ser_tiques_temporal_valid`.

El PDF de tiques indica que `INFORMACIÓN NO DISPONIBLE` en `distintivo` significa que el sistema no ha requerido conocer ni almacenar la categoría ambiental. El propio documento cita como ejemplo los vehículos comerciales, que obtienen tique sin coste por realizar pagos mensuales o anuales.

Por tanto, se cruza `tipo_zona` con `distintivo` para comprobar si las categorías `COMERCIALES` y `TALLERES` reflejan presión observada en tiques. Esta comprobación no modifica todavía `ser_autorizaciones`; aporta evidencia para decidir más adelante cómo construir el proxy de presión no observada.


In [80]:
info_no_disponible_by_tipo_zona = con.execute('''
    SELECT
        tipo_zona,
        COUNT(*) AS n_rows,
        SUM(CASE
            WHEN upper(distintivo) LIKE '%NO DISPONIBLE%'
            THEN 1 ELSE 0
        END) AS n_distintivo_info_no_disponible
    FROM ser_tiques_temporal_valid
    GROUP BY tipo_zona
    ORDER BY n_rows DESC
''').df()

info_no_disponible_by_tipo_zona["pct_info_no_disponible"] = (
    info_no_disponible_by_tipo_zona["n_distintivo_info_no_disponible"]
    / info_no_disponible_by_tipo_zona["n_rows"]
    * 100
).round(6)

commercial_workshop_evidence = (
    info_no_disponible_by_tipo_zona
    .loc[info_no_disponible_by_tipo_zona["tipo_zona"].isin(["COMERCIALES", "TALLERES"])]
    .copy()
)

commercial_workshop_evidence["lectura"] = [
    "presion_observable_en_tiques" if z in ["COMERCIALES", "TALLERES"] else "revisar"
    for z in commercial_workshop_evidence["tipo_zona"]
]

display(info_no_disponible_by_tipo_zona)
display(commercial_workshop_evidence)

info_no_disponible_by_tipo_zona.to_csv(
    REPORTS_TABLES / "ser_tiques_info_no_disponible_by_tipo_zona.csv",
    index=False
)

commercial_workshop_evidence.to_csv(
    REPORTS_TABLES / "ser_tiques_commercial_workshop_evidence.csv",
    index=False
)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,tipo_zona,n_rows,n_distintivo_info_no_disponible,pct_info_no_disponible
0,VERDE,80616082,548084.0,0.679869
1,AZUL,57745587,318235.0,0.551098
2,COMERCIALES,8638084,1358342.0,15.725038
3,USO DISUASORIO,1554107,10201.0,0.656390
4,AZUL SANITARIA,926057,5819.0,0.628363
5,ALTA ROTACION,586056,2887.0,0.492615
6,TALLERES,71918,71918.0,100.000000


,tipo_zona,n_rows,n_distintivo_info_no_disponible,pct_info_no_disponible,lectura
2,COMERCIALES,8638084,1358342.0,15.725038,presion_observable_en_tiques
6,TALLERES,71918,71918.0,100.000000,presion_observable_en_tiques


**Lectura/decisión.**

La presencia explícita de `COMERCIALES` y `TALLERES` dentro de `ser_tiques` permite tratarlos como presión observada en superficie. Este resultado es especialmente claro en talleres, donde el distintivo aparece como `INFORMACIÓN NO DISPONIBLE` de forma sistemática, y también es relevante en comerciales por su volumen.

Dado que no existe un identificador común entre autorizaciones y tiques, no se puede verificar autorización a autorización. Metodológicamente se asume que las categorías `COMERCIALES` y `TALLERES` observadas en tiques capturan esa presión en superficie y, por tanto, no se incorporarán de nuevo como presión no observada en el proxy.

La implicación metodológica no se aplica destruyendo la fuente limpia de autorizaciones, sino en la fase de construcción del proxy.


### 7.12. Diagnóstico de límites de duración por `tipo_zona`

Esta comprobación se realiza sobre `ser_tiques_temporal_valid`, no sobre la tabla base completa. Es decir, solo se analizan registros que ya superaron:

- fechas no nulas;
- `fecha_fin >= fecha_inicio`;
- `fecha_inicio` dentro de la ventana global cubierta;
- duración positiva.

Los límites usados proceden de fuentes oficiales del Ayuntamiento de Madrid:

- SER general — horarios, zona verde, zona azul, alta rotación, disuasorio y ámbito hospitalario/socio-sanitario:  
  https://www.madrid.es/portales/munimadrid/es/Inicio/Movilidad-y-transportes/Servicio-de-Estacionamiento-Regulado-SER-Horario-delimitacion-territorial-y-ambitos-diferenciados-/?vgnextchannel=220e31d3b28fe410VgnVCM1000000b205a0aRCRD&vgnextfmt=default&vgnextoid=6f18e4ce78dd6410VgnVCM1000000b205a0aRCRD

- Autorizaciones SER — Portal de Transparencia: comerciales y talleres:  
  https://transparencia.madrid.es/portales/transparencia/es/Transparencia-por-sectores/Movilidad/Estacionamiento/Autorizaciones-del-Servicio-de-Estacionamiento-Regulado-SER/?vgnextchannel=f49a709f90d1d510VgnVCM2000001f4a900aRCRD&vgnextfmt=default&vgnextoid=0b48e23be8aca910VgnVCM200000f921e388RCRD

- Autorizaciones SER — Datos Abiertos: comerciales y talleres:  
  https://datos.madrid.es/dataset/300088-0-ser-autorizaciones

Los umbrales son acumulativos: por ejemplo, `n_supera_limite_mas_120` es subconjunto de `n_supera_limite_normativo`. Por ello, se muestran solo los umbrales necesarios para tomar decisiones: exceso normativo, exceso fuerte superior a dos horas, duración superior a 24 h, duración superior a 48 h y duración superior a 7 días.


In [81]:
duration_limits = pd.DataFrame([
    {"tipo_zona": "VERDE", "limite_minutos_diagnostico": 120, "criterio": "verde_no_residente"},
    {"tipo_zona": "AZUL", "limite_minutos_diagnostico": 240, "criterio": "azul_general"},
    {"tipo_zona": "ALTA ROTACION", "limite_minutos_diagnostico": 45, "criterio": "alta_rotacion_general"},
    {"tipo_zona": "USO DISUASORIO", "limite_minutos_diagnostico": 720, "criterio": "ambito_disuasorio_12h_servicio"},
    {"tipo_zona": "AZUL SANITARIA", "limite_minutos_diagnostico": 240, "criterio": "ambito_socio_sanitario"},
    {"tipo_zona": "COMERCIALES", "limite_minutos_diagnostico": 480, "criterio": "comercial_limite_superior_8h"},
    {"tipo_zona": "TALLERES", "limite_minutos_diagnostico": 300, "criterio": "talleres_5h"},
])

con.register("duration_limits", duration_limits)

duration_limit_diagnostic_full = con.execute('''
    WITH joined AS (
        SELECT
            t.tipo_zona,
            t.duracion_minutos,
            t.importe_tique,
            l.limite_minutos_diagnostico,
            l.criterio
        FROM ser_tiques_temporal_valid t
        LEFT JOIN duration_limits l
            ON t.tipo_zona = l.tipo_zona
    )
    SELECT
        tipo_zona,
        criterio,
        limite_minutos_diagnostico,
        COUNT(*) AS n_rows_validos,
        SUM(CASE WHEN duracion_minutos > limite_minutos_diagnostico THEN 1 ELSE 0 END) AS n_supera_limite_normativo,
        SUM(CASE WHEN duracion_minutos > limite_minutos_diagnostico + 120 THEN 1 ELSE 0 END) AS n_supera_limite_mas_120,
        SUM(CASE WHEN duracion_minutos > 1440 THEN 1 ELSE 0 END) AS n_supera_24h,
        SUM(CASE WHEN duracion_minutos > 2880 THEN 1 ELSE 0 END) AS n_supera_48h,
        SUM(CASE WHEN duracion_minutos > 10080 THEN 1 ELSE 0 END) AS n_supera_7d,
        MAX(duracion_minutos) AS max_duracion_minutos,
        ROUND(AVG(importe_tique), 6) AS importe_medio,
        ROUND(AVG(CASE WHEN duracion_minutos > limite_minutos_diagnostico + 120 THEN importe_tique END), 6) AS importe_medio_supera_limite_mas_120
    FROM joined
    GROUP BY tipo_zona, criterio, limite_minutos_diagnostico
    ORDER BY n_rows_validos DESC
''').df()

for col in [
    "n_supera_limite_normativo",
    "n_supera_limite_mas_120",
    "n_supera_24h",
    "n_supera_48h",
    "n_supera_7d",
]:
    duration_limit_diagnostic_full[f"pct_{col}"] = (
        duration_limit_diagnostic_full[col]
        / duration_limit_diagnostic_full["n_rows_validos"]
        * 100
    ).round(6)

duration_limit_diagnostic = duration_limit_diagnostic_full.copy()

duration_limit_diagnostic_display = duration_limit_diagnostic_full[[
    "tipo_zona",
    "limite_minutos_diagnostico",
    "n_rows_validos",
    "n_supera_limite_normativo",
    "pct_n_supera_limite_normativo",
    "n_supera_limite_mas_120",
    "pct_n_supera_limite_mas_120",
    "n_supera_24h",
    "n_supera_48h",
    "n_supera_7d",
    "max_duracion_minutos",
]].copy()

display(duration_limits)
display(duration_limit_diagnostic_display)

con.execute('''
    CREATE OR REPLACE VIEW ser_tiques_final_valid AS
    SELECT *
    FROM ser_tiques_temporal_valid
    WHERE duracion_minutos <= 10080
''')

n_temporal_valid = con.execute("SELECT COUNT(*) FROM ser_tiques_temporal_valid").fetchone()[0]
n_final_valid = con.execute("SELECT COUNT(*) FROM ser_tiques_final_valid").fetchone()[0]

print("Filas temporalmente válidas:", f"{n_temporal_valid:,}")
print("Filas finales tras excluir > 7 días:", f"{n_final_valid:,}")
print("Filas excluidas por duración > 7 días:", f"{n_temporal_valid - n_final_valid:,}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,tipo_zona,limite_minutos_diagnostico,criterio
0,VERDE,120,verde_no_residente
1,AZUL,240,azul_general
2,ALTA ROTACION,45,alta_rotacion_general
3,USO DISUASORIO,720,ambito_disuasorio_12h_servicio
4,AZUL SANITARIA,240,ambito_socio_sanitario
5,COMERCIALES,480,comercial_limite_superior_8h
6,TALLERES,300,talleres_5h


,tipo_zona,limite_minutos_diagnostico,n_rows_validos,n_supera_limite_normativo,pct_n_supera_limite_normativo,n_supera_limite_mas_120,pct_n_supera_limite_mas_120,n_supera_24h,n_supera_48h,n_supera_7d,max_duracion_minutos
0,VERDE,120,80616082,4584030.0,5.686248,4584030.0,5.686248,1252920.0,86409.0,7.0,281529
1,AZUL,240,57745587,4155920.0,7.196948,4155920.0,7.196948,945274.0,72263.0,0.0,5640
2,COMERCIALES,480,8638084,928.0,0.010743,0.0,0.000000,0.0,0.0,0.0,493
3,USO DISUASORIO,720,1554107,55602.0,3.577746,19278.0,1.240455,13261.0,1741.0,0.0,5760
4,AZUL SANITARIA,240,926057,33202.0,3.585308,33202.0,3.585308,7736.0,650.0,0.0,5640
5,ALTA ROTACION,45,586056,17978.0,3.067625,17978.0,3.067625,4972.0,280.0,0.0,5445
6,TALLERES,300,71918,11.0,0.015295,0.0,0.000000,0.0,0.0,0.0,301


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Filas temporalmente válidas: 150,137,891
Filas finales tras excluir > 7 días: 150,137,884
Filas excluidas por duración > 7 días: 7


**Lectura/decisión.**

Los límites por `tipo_zona` sirven para dimensionar anomalías de duración, pero no se aplican como filtro definitivo en esta limpieza individual. Los umbrales son acumulativos: los registros que superan el límite más 120 minutos también forman parte de los que superan el límite normativo.

En VERDE y AZUL se observa que los excesos no son residuales: los registros que superan el límite normativo también superan el límite en más de 120 minutos. Además, el volumen de casos superiores a 24 h y 48 h es demasiado elevado para tratarlos como errores intrínsecos aislados sin cruzar calendario. Su magnitud sugiere una interacción con régimen horario, días sin servicio, continuidad de validez o reglas operativas.

Por tanto, estos casos no se filtran en este notebook. Se mantienen para análisis posterior y se revisarán en el notebook de joins SER con calendario, horario efectivo, importe y, si procede, reglas espaciales.

Sí se aplica un filtro intrínseco muy conservador: se excluyen los tiques con `duracion_minutos > 10080`, equivalente a más de 7 días. Este umbral no pretende representar una regla SER, sino retirar outliers temporales extremos que no son compatibles con un intervalo razonable de estacionamiento regulado.

No se persiste un subconjunto de registros excedidos en esta fase para no duplicar datos. En el notebook de joins SER se reconstruirá ese subconjunto desde `final_clean_parts` y se cruzará con calendario, horario efectivo, importe y, si procede, reglas espaciales.


### 7.13. Duplicados exactos sobre columnas operativas

Se cuantifican duplicados exactos sobre `ser_tiques_final_valid`, es decir, tras excluir registros con fechas inválidas, fuera de ventana global, duración no positiva y outliers temporales superiores a 7 días.

La clave de duplicación usa las 12 columnas operativas originales. No se usan columnas artificiales ni de trazabilidad como parte de la clave.


In [82]:
if RUN_DUPLICATE_CHECK:
    duplicate_summary_by_period = con.execute('''
        WITH grouped AS (
            SELECT
                periodo_inicio,
                matricula_parquimetro,
                fecha_operacion,
                fecha_inicio,
                fecha_fin,
                cod_distrito,
                distrito,
                cod_barrio,
                barrio,
                tipo_zona,
                distintivo,
                minutos_tique,
                importe_tique,
                COUNT(*) AS cnt
            FROM ser_tiques_final_valid
            GROUP BY
                periodo_inicio,
                matricula_parquimetro,
                fecha_operacion,
                fecha_inicio,
                fecha_fin,
                cod_distrito,
                distrito,
                cod_barrio,
                barrio,
                tipo_zona,
                distintivo,
                minutos_tique,
                importe_tique
            HAVING COUNT(*) > 1
        )
        SELECT
            periodo_inicio,
            COUNT(*) AS n_duplicate_keys,
            SUM(cnt - 1) AS n_duplicate_extra_rows,
            MAX(cnt) AS max_repetition_same_key
        FROM grouped
        GROUP BY periodo_inicio
        ORDER BY periodo_inicio
    ''').df()

    display(duplicate_summary_by_period)

    duplicate_summary_by_period.to_csv(
        REPORTS_TABLES / "ser_tiques_duplicate_summary_by_period.csv",
        index=False
    )
else:
    duplicate_summary_by_period = pd.DataFrame()
    print("Chequeo de duplicados exactos omitido: RUN_DUPLICATE_CHECK = False")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,periodo_inicio,n_duplicate_keys,n_duplicate_extra_rows,max_repetition_same_key
0,2023_q1,5332,5332.0,2
1,2023_q2,561,561.0,2
2,2023_q3,1,1.0,2


**Lectura/decisión.**

Los duplicados exactos se documentan, pero no se eliminan. Sin identificador único de tique no puede demostrarse de forma concluyente que todos los duplicados sean errores técnicos. Además, si el volumen es marginal, su impacto sobre agregaciones posteriores será reducido.


### 7.14. Escritura de salida limpia final de tiques

A partir de `ser_tiques_final_valid` se escribe una salida final local en:

`data/interim/ser/ser_tiques/final_clean_parts/`

Esta salida:

- elimina columnas no usadas como variables finales (`fecha_operacion`, `distintivo`, `minutos_tique`);
- materializa `duracion_minutos` aplicando la misma regla definida y validada en la sección 7.9;
- conserva solo registros con fechas válidas, duración positiva, fecha de inicio dentro de la ventana global cubierta y duración no superior a 7 días;
- particiona físicamente por `periodo_inicio`, no por `periodo_raw`, para que los registros reasignables queden en el trimestre analítico correspondiente;
- no aplica todavía filtros por horario SER, calendario, parquímetro, calle o geometría.

En esta ejecución se fuerza la regeneración de `final_clean_parts` para asegurar que la salida refleja las reglas actualizadas.


In [83]:
import math
import pyarrow.parquet as pq

FINAL_CLEAN_PARTS_DIR = ROOT / "data/interim/ser/ser_tiques/final_clean_parts"

# El manifiesto final se mantiene solo en memoria.
# Si existe un CSV de ejecuciones anteriores, se elimina para no persistir artefactos no usados.
FINAL_MANIFEST_PATH = REPORTS_TABLES / "ser_tiques_final_clean_parts_manifest.csv"
if FINAL_MANIFEST_PATH.exists():
    FINAL_MANIFEST_PATH.unlink()
    print("Manifiesto final CSV eliminado:", FINAL_MANIFEST_PATH.relative_to(ROOT))

RECREATE_FINAL_PARTS = False

def extract_periodo_inicio_from_partition_path(path: Path) -> str:
    for part in path.parts:
        if part.startswith("periodo_inicio="):
            return part.split("=", 1)[1]
    raise ValueError(f"No se puede extraer periodo_inicio de la ruta: {path}")

def build_final_manifest_from_files(files: list[Path]) -> pd.DataFrame:
    rows = []
    for i, path in enumerate(files):
        metadata = pq.ParquetFile(path).metadata
        filename = path.name
        periodo_raw_origen = None
        if "__raw_" in filename:
            periodo_raw_origen = filename.split("__raw_", 1)[1].replace(".parquet", "")

        rows.append({
            "periodo_inicio": extract_periodo_inicio_from_partition_path(path),
            "periodo_raw_origen": periodo_raw_origen,
            "part_number": i,
            "n_rows": metadata.num_rows,
            "n_columns": metadata.num_columns,
            "output_path": str(path.relative_to(ROOT)),
            "output_size_mb": round(path.stat().st_size / (1024 ** 2), 3),
        })
    return pd.DataFrame(rows)

final_files_existing = parquet_files_under(FINAL_CLEAN_PARTS_DIR)

if RECREATE_FINAL_PARTS and FINAL_CLEAN_PARTS_DIR.exists():
    shutil.rmtree(FINAL_CLEAN_PARTS_DIR)
    final_files_existing = []

expected_final_rows = con.execute('''
    SELECT COUNT(*) FROM ser_tiques_final_valid
''').fetchone()[0]

if final_files_existing:
    print("Se reutiliza la salida final existente:", FINAL_CLEAN_PARTS_DIR.relative_to(ROOT))
    ser_tiques_final_parts_manifest = build_final_manifest_from_files(final_files_existing)

else:
    print("Se escribe salida final limpia desde base_clean_parts.")

    if FINAL_CLEAN_PARTS_DIR.exists():
        shutil.rmtree(FINAL_CLEAN_PARTS_DIR)

    FINAL_CLEAN_PARTS_DIR.mkdir(parents=True, exist_ok=True)

    final_manifest_rows = []
    total_excluded_final = 0
    write_counter = 0

    base_files = parquet_files_under(BASE_CLEAN_PARTS_DIR)

    for i, base_path in enumerate(base_files):
        periodo_raw = extract_periodo_from_partition_path(base_path)

        df = pd.read_parquet(base_path, engine=PARQUET_ENGINE)

        for col in ["fecha_inicio", "fecha_fin"]:
            df[col] = pd.to_datetime(df[col], errors="coerce")

        duracion_segundos = (df["fecha_fin"] - df["fecha_inicio"]).dt.total_seconds()

        df["periodo_inicio"] = pd.NA
        in_global_window = (
            df["fecha_inicio"].ge(pd.Timestamp(MIN_GLOBAL_DATE))
            & df["fecha_inicio"].lt(pd.Timestamp(MAX_GLOBAL_DATE_EXCLUSIVE))
        )
        df.loc[in_global_window, "periodo_inicio"] = (
            df.loc[in_global_window, "fecha_inicio"].dt.year.astype(str)
            + "_q"
            + df.loc[in_global_window, "fecha_inicio"].dt.quarter.astype(str)
        )

        valid_mask = (
            df["fecha_inicio"].notna()
            & df["fecha_fin"].notna()
            & df["fecha_fin"].ge(df["fecha_inicio"])
            & duracion_segundos.gt(0)
            & df["periodo_inicio"].notna()
        )

        duracion_minutos = (duracion_segundos / 60.0).apply(
            lambda x: math.ceil(x) if pd.notna(x) else pd.NA
        )

        valid_mask = valid_mask & duracion_minutos.le(10080)

        total_excluded_final += int((~valid_mask).sum())

        out = df.loc[valid_mask, [
            "periodo_inicio",
            "matricula_parquimetro",
            "fecha_inicio",
            "fecha_fin",
            "cod_distrito",
            "distrito",
            "cod_barrio",
            "barrio",
            "tipo_zona",
            "importe_tique",
        ]].copy()

        out["duracion_minutos"] = (
            duracion_minutos.loc[valid_mask]
            .astype("Int64")
        )

        out["cod_distrito"] = pd.to_numeric(out["cod_distrito"], errors="coerce").astype("Int64")
        out["cod_barrio"] = pd.to_numeric(out["cod_barrio"], errors="coerce").astype("Int64")

        out = out[["periodo_inicio"] + SER_TIQUES_FINAL_COLUMNS].copy()

        if out.empty:
            continue

        for periodo_inicio, out_period in out.groupby("periodo_inicio", dropna=False):
            period_dir = FINAL_CLEAN_PARTS_DIR / f"periodo_inicio={periodo_inicio}"
            period_dir.mkdir(parents=True, exist_ok=True)

            output_path = period_dir / f"part_{write_counter:06d}__raw_{periodo_raw}.parquet"
            out_to_write = out_period[SER_TIQUES_FINAL_COLUMNS].copy()

            out_to_write.to_parquet(
                output_path,
                engine=PARQUET_ENGINE,
                compression="snappy",
                index=False
            )

            final_manifest_rows.append({
                "periodo_inicio": periodo_inicio,
                "periodo_raw_origen": periodo_raw,
                "part_number": write_counter,
                "n_rows": len(out_to_write),
                "n_columns": out_to_write.shape[1],
                "output_path": str(output_path.relative_to(ROOT)),
                "output_size_mb": round(output_path.stat().st_size / (1024 ** 2), 3),
            })

            write_counter += 1

        if (i + 1) % 50 == 0:
            print(f"Partes base procesadas: {i + 1:,} / {len(base_files):,}")

    ser_tiques_final_parts_manifest = pd.DataFrame(final_manifest_rows)

    print("Filas excluidas en escritura final:", f"{total_excluded_final:,}")

final_rows = int(ser_tiques_final_parts_manifest["n_rows"].sum())

final_write_checks = pd.DataFrame([
    {
        "check": "filas_esperadas_vs_escritas",
        "valor": final_rows,
        "esperado": expected_final_rows,
        "ok": final_rows == expected_final_rows,
    },
    {
        "check": "n_partes_finales",
        "valor": len(ser_tiques_final_parts_manifest),
        "esperado": ">= 1",
        "ok": len(ser_tiques_final_parts_manifest) >= 1,
    },
    {
        "check": "n_columnas_finales",
        "valor": int(ser_tiques_final_parts_manifest["n_columns"].max()),
        "esperado": len(SER_TIQUES_FINAL_COLUMNS),
        "ok": ser_tiques_final_parts_manifest["n_columns"].eq(len(SER_TIQUES_FINAL_COLUMNS)).all(),
    },
])

display(final_write_checks)

print("Filas esperadas según ser_tiques_final_valid:", f"{expected_final_rows:,}")
print("Filas escritas en final_clean_parts:", f"{final_rows:,}")
print("Coinciden filas esperadas y escritas:", expected_final_rows == final_rows)

final_sample_path = ROOT / ser_tiques_final_parts_manifest.iloc[0]["output_path"]
final_sample = pd.read_parquet(final_sample_path, engine=PARQUET_ENGINE)

print("Muestra final re-leída:", final_sample_path.relative_to(ROOT))
print("Shape muestra final:", final_sample.shape)
print("Columnas finales correctas:", list(final_sample.columns) == SER_TIQUES_FINAL_COLUMNS)

display(
    final_sample
    .dtypes
    .astype(str)
    .reset_index()
    .rename(columns={"index": "columna", 0: "dtype"})
)


Manifiesto final CSV eliminado: reports/tables/ser_tiques_final_clean_parts_manifest.csv


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Se escribe salida final limpia desde base_clean_parts.
Partes base procesadas: 50 / 606
Partes base procesadas: 100 / 606
Partes base procesadas: 150 / 606
Partes base procesadas: 200 / 606
Partes base procesadas: 250 / 606
Partes base procesadas: 300 / 606
Partes base procesadas: 350 / 606
Partes base procesadas: 400 / 606
Partes base procesadas: 450 / 606
Partes base procesadas: 500 / 606
Partes base procesadas: 550 / 606
Partes base procesadas: 600 / 606
Filas excluidas en escritura final: 2,040


,check,valor,esperado,ok
0,filas_esperadas_vs_escritas,150137884,150137884,True
1,n_partes_finales,1002,>= 1,True
2,n_columnas_finales,10,10,True


Filas esperadas según ser_tiques_final_valid: 150,137,884
Filas escritas en final_clean_parts: 150,137,884
Coinciden filas esperadas y escritas: True
Muestra final re-leída: data/interim/ser/ser_tiques/final_clean_parts/periodo_inicio=2023_q1/part_000000__raw_2023_q1.parquet
Shape muestra final: (249976, 10)
Columnas finales correctas: True


,columna,dtype
0,matricula_parquimetro,string
1,fecha_inicio,datetime64[us]
2,fecha_fin,datetime64[us]
3,duracion_minutos,Int64
4,cod_distrito,Int64
5,distrito,string
6,cod_barrio,Int64
7,barrio,string
8,tipo_zona,string
9,importe_tique,float64


### 7.15. Resumen final de validaciones intrínsecas

Se construye una tabla final de decisión con las incidencias principales, su volumen y la acción adoptada en este notebook.


In [84]:
total_rows = int(n_manifest_rows)

temporal_row = temporal_checks_raw.iloc[0].to_dict()
duration_zero_row = duration_zero_check.iloc[0].to_dict()
duration_row = duration_comparison.iloc[0].to_dict()
key_row = key_nulls_global.iloc[0].to_dict()

if RUN_DUPLICATE_CHECK and not duplicate_summary_by_period.empty:
    duplicate_extra_rows = int(duplicate_summary_by_period["n_duplicate_extra_rows"].sum())
else:
    duplicate_extra_rows = 0

final_rows = int(ser_tiques_final_parts_manifest["n_rows"].sum())
excluded_final = total_rows - final_rows

n_commercial_workshop = int(commercial_workshop_evidence["n_rows"].sum())
n_limit_excess = int(duration_limit_diagnostic["n_supera_limite_normativo"].sum())
n_limit_excess_120 = int(duration_limit_diagnostic["n_supera_limite_mas_120"].sum())
n_gt_24h = int(duration_limit_diagnostic["n_supera_24h"].sum())
n_gt_48h = int(duration_limit_diagnostic["n_supera_48h"].sum())
n_gt_7d = int(duration_limit_diagnostic["n_supera_7d"].sum())

overview_items = [
    {
        "validacion": "filas_manifest_vs_parquet",
        "n_afectadas": 0 if n_parquet_rows == n_manifest_rows else abs(int(n_parquet_rows) - int(n_manifest_rows)),
        "decision": "ok",
        "accion": "se conserva salida base",
    },
    {
        "validacion": "columnas_base",
        "n_afectadas": 0 if actual_columns == expected_columns else len(set(expected_columns).symmetric_difference(actual_columns)),
        "decision": "ok",
        "accion": "se trabaja con 12 columnas operativas",
    },
    {
        "validacion": "fechas_nulas",
        "n_afectadas": int(temporal_row["n_fecha_operacion_null"] + temporal_row["n_fecha_inicio_null"] + temporal_row["n_fecha_fin_null"]),
        "decision": "ok si 0",
        "accion": "sin exclusión adicional si no hay nulos",
    },
    {
        "validacion": "fecha_fin_menor_inicio",
        "n_afectadas": int(temporal_row["n_fecha_fin_menor_inicio"]),
        "decision": "excluir del final",
        "accion": "no representa intervalo temporal válido",
    },
    {
        "validacion": "fecha_inicio_fuera_ventana_global",
        "n_afectadas": int(temporal_row["n_fecha_inicio_fuera_ventana_global"]),
        "decision": "excluir del final",
        "accion": "queda fuera de la cobertura temporal de la fuente",
    },
    {
        "validacion": "fecha_inicio_reasignable_a_otro_periodo",
        "n_afectadas": int(temporal_row["n_fecha_inicio_reasignable_a_otro_periodo"]),
        "decision": "conservar_reasignando",
        "accion": "se particiona por periodo_inicio",
    },
    {
        "validacion": "fecha_fin_fuera_ventana_global",
        "n_afectadas": int(temporal_row["n_fecha_fin_fuera_ventana_global"]),
        "decision": "diagnostico",
        "accion": "no se filtra si fecha_inicio pertenece a la ventana cubierta",
    },
    {
        "validacion": "intervalo_cruza_trimestre_inicio",
        "n_afectadas": int(temporal_row["n_intervalo_cruza_trimestre_inicio"]),
        "decision": "diagnostico",
        "accion": "revisar si el panel horario reparte intervalos por hora",
    },
    {
        "validacion": "duracion_cero_calculada",
        "n_afectadas": int(duration_zero_row["n_duracion_cero_calculada"]),
        "decision": "excluir del final",
        "accion": "no representa intervalo de estacionamiento positivo",
    },
    {
        "validacion": "diferencia_minutos_tique_vs_duracion",
        "n_afectadas": int(duration_row["n_dif_ceil"]),
        "decision": "no filtrar por esta variable",
        "accion": "se descarta minutos_tique y se materializa duracion_minutos desde fechas válidas",
    },
    {
        "validacion": "nulos_claves_operativas",
        "n_afectadas": int(
            key_row["n_cod_distrito_null"]
            + key_row["n_cod_barrio_null"]
            + key_row["n_barrio_null"]
            + key_row["n_matricula_parquimetro_null"]
        ),
        "decision": "ok si 0",
        "accion": "habilita joins posteriores por barrio y parquímetro",
    },
    {
        "validacion": "nulos_distrito_textual",
        "n_afectadas": int(key_row["n_distrito_null"]),
        "decision": "mantener",
        "accion": "campo descriptivo reconstruible desde cod_distrito",
    },
    {
        "validacion": "tipo_zona_comerciales_talleres",
        "n_afectadas": n_commercial_workshop,
        "decision": "presion_observable",
        "accion": "no tratarlos automáticamente como presión no observada en proxy posterior",
    },
    {
        "validacion": "supera_limite_normativo_diagnostico",
        "n_afectadas": n_limit_excess,
        "decision": "diagnostico",
        "accion": "no filtrar hasta validar calendario/horario/señalización",
    },
    {
        "validacion": "supera_limite_normativo_mas_120",
        "n_afectadas": n_limit_excess_120,
        "decision": "diagnostico_fuerte",
        "accion": "revisar en join_ser con calendario e importe",
    },
    {
        "validacion": "duracion_superior_24h",
        "n_afectadas": n_gt_24h,
        "decision": "outlier_temporal",
        "accion": "cuantificado; pendiente decisión con calendario",
    },
    {
        "validacion": "duracion_superior_48h",
        "n_afectadas": n_gt_48h,
        "decision": "outlier_temporal_extremo",
        "accion": "cuantificado; pendiente decisión con calendario",
    },
    {
        "validacion": "duracion_superior_7d",
        "n_afectadas": n_gt_7d,
        "decision": "excluir",
        "accion": "outlier temporal extremo incompatible con intervalo razonable SER",
    },
    {
        "validacion": "duplicados_exactos_extra",
        "n_afectadas": duplicate_extra_rows,
        "decision": "mantener",
        "accion": "sin id único de tique; volumen marginal",
    },
    {
        "validacion": "filas_excluidas_final_clean_parts",
        "n_afectadas": excluded_final,
        "decision": "filtrado final aplicado",
        "accion": "745 fechas fuera/incoherentes + 1288 duración cero + 7 duración > 7 días",
    },
]

ser_tiques_validation_overview = pd.DataFrame(overview_items)
ser_tiques_validation_overview["pct_afectado"] = (
    ser_tiques_validation_overview["n_afectadas"] / total_rows * 100
).round(6)

display(ser_tiques_validation_overview)

ser_tiques_validation_overview.to_csv(
    REPORTS_TABLES / "ser_tiques_validation_overview.csv",
    index=False
)

print("Validaciones intrínsecas completadas.")
print("Filas base validadas:", f"{total_rows:,}")
print("Filas finales escritas:", f"{final_rows:,}")
print("Pendiente: validaciones cruzadas en notebook join_ser.")


,validacion,n_afectadas,decision,accion,pct_afectado
0,filas_manifest_vs_parquet,0,ok,se conserva salida base,0.000000
1,columnas_base,0,ok,se trabaja con 12 columnas operativas,0.000000
2,fechas_nulas,0,ok si 0,sin exclusión adicional si no hay nulos,0.000000
3,fecha_fin_menor_inicio,91,excluir del final,no representa intervalo temporal válido,0.000061
4,fecha_inicio_fuera_ventana_global,654,excluir del final,queda fuera de la cobertura temporal de la fuente,0.000436
5,fecha_inicio_reasignable_a_otro_periodo,23415,conservar_reasignando,se particiona por periodo_inicio,0.015595
6,fecha_fin_fuera_ventana_global,9632,diagnostico,no se filtra si fecha_inicio pertenece a la ventana cubierta,0.006415
7,intervalo_cruza_trimestre_inicio,111741,diagnostico,revisar si el panel horario reparte intervalos por hora,0.074425
8,duracion_cero_calculada,1288,excluir del final,no representa intervalo de estacionamiento positivo,0.000858
9,diferencia_minutos_tique_vs_duracion,9611978,no filtrar por esta variable,se descarta minutos_tique y se materializa duracion_minutos desde fechas válidas,6.402013


Validaciones intrínsecas completadas.
Filas base validadas: 150,139,924
Filas finales escritas: 150,137,884
Pendiente: validaciones cruzadas en notebook join_ser.


### Lectura global de las validaciones intrínsecas

La limpieza individual de `ser_tiques` queda orientada a producir una fuente base de eventos de tique con columnas temporales y espaciales utilizables. La escritura se valida mediante manifiesto y lectura Parquet, y las reglas de exclusión aplicadas se limitan a problemas temporales intrínsecos: fechas inconsistentes, registros fuera de la ventana global cubierta, duraciones no positivas y outliers temporales extremos superiores a 7 días.

La duración final se deriva de `fecha_inicio` y `fecha_fin`, porque estas columnas representan el comienzo y fin de validez del tique. `minutos_tique` se utiliza como contraste, pero no se conserva como variable final debido a las discrepancias detectadas frente a la duración calculada.

`fecha_operacion` y `distintivo` tampoco se conservan en la salida final. La primera no define el intervalo de validez del estacionamiento; la segunda se utiliza como diagnóstico para interpretar comerciales y talleres, pero no es necesaria para el modelado principal de dificultad SER.

Los límites por `tipo_zona` se documentan con fuentes oficiales y se usan como diagnóstico sobre registros temporalmente válidos. No se aplican como filtro automático porque las reglas finas dependen de calendario, horario efectivo, señalización, parquímetros, calles y geometría. Solo se excluyen los casos extremos con duración superior a 7 días.

Conclusión: `ser_tiques` queda preparado como fuente limpia individual para fases posteriores, pero la validación completa del régimen SER observable se realizará más adelante al integrar calendario, parquímetros, calles/plazas y capa cartográfica.


## 8. Cierre

Este notebook deja preparada la fuente `ser_tiques` en dos niveles locales:

- `base_clean_parts`: salida base con las 12 columnas operativas normalizadas.
- `final_clean_parts`: salida limpia final con columnas temporales, espaciales y duración materializada.

Los tiques limpios constituyen la señal temporal principal de demanda pagada. En el flujo posterior, la unidad espacial principal es el barrio y el panel final se construye como barrio × intervalo temporal, con una granularidad seleccionada de 30 minutos.

La escala de calle o parquímetro se conserva únicamente como análisis complementario cuando existe un identificador físico verificable. Los tiques preparados alimentan posteriormente la construcción de la base SER y del panel barrio–intervalo.
